In [1]:
import os
import numpy as np
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

# --------------------------------------------------
# Project paths
# --------------------------------------------------

PROJECT_DIR = r"C:\Users\RAJINI\OneDrive\Desktop\AI-Soil-Analytics"

TRAIN_DIR = os.path.join(
    PROJECT_DIR, "data", "images", "split", "train"
)

VAL_DIR = os.path.join(
    PROJECT_DIR, "data", "images", "split", "validation"
)

TEST_DIR = os.path.join(
    PROJECT_DIR, "data", "images", "split", "test"
)

# --------------------------------------------------
# Image settings
# --------------------------------------------------

IMG_SIZE = (224, 224)
BATCH_SIZE = 8
SEED = 42

# --------------------------------------------------
# Check paths
# --------------------------------------------------

print("Train directory exists:", os.path.exists(TRAIN_DIR))
print("Validation directory exists:", os.path.exists(VAL_DIR))
print("Test directory exists:", os.path.exists(TEST_DIR))

In [2]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
)

class_names = train_ds.class_names
NUM_CLASSES = len(class_names)

print("\nClasses:", class_names)
print("Number of classes:", NUM_CLASSES)

In [3]:
print("Training batches:", len(train_ds))
print("Validation batches:", len(val_ds))
print("Test batches:", len(test_ds))

In [4]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dropout, Dense
from tensorflow.keras.models import Sequential
from tensorflow.keras.optimizers import Adam

base_model = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

# Freeze pretrained ResNet-50
base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

In [5]:
import time

train_features = []
train_labels = []

start_time = time.time()

for images, labels in train_ds:
    features = base_model(images, training=False)
    features = tf.keras.layers.GlobalAveragePooling2D()(features)

    train_features.append(features.numpy())
    train_labels.append(labels.numpy())

train_features = np.concatenate(train_features, axis=0)
train_labels = np.concatenate(train_labels, axis=0)

elapsed = time.time() - start_time

print("Training features:", train_features.shape)
print("Training labels:", train_labels.shape)
print(f"Extraction time: {elapsed / 60:.2f} minutes")

In [6]:
val_features = []
val_labels = []

start_time = time.time()

for images, labels in val_ds:
    features = base_model(images, training=False)
    features = tf.keras.layers.GlobalAveragePooling2D()(features)

    val_features.append(features.numpy())
    val_labels.append(labels.numpy())

val_features = np.concatenate(val_features, axis=0)
val_labels = np.concatenate(val_labels, axis=0)

elapsed = time.time() - start_time

print("Validation features:", val_features.shape)
print("Validation labels:", val_labels.shape)
print(f"Extraction time: {elapsed / 60:.2f} minutes")

In [7]:
test_features = []
test_labels = []

start_time = time.time()

for images, labels in test_ds:
    features = base_model(images, training=False)
    features = tf.keras.layers.GlobalAveragePooling2D()(features)

    test_features.append(features.numpy())
    test_labels.append(labels.numpy())

test_features = np.concatenate(test_features, axis=0)
test_labels = np.concatenate(test_labels, axis=0)

elapsed = time.time() - start_time

print("Test features:", test_features.shape)
print("Test labels:", test_labels.shape)
print(f"Extraction time: {elapsed / 60:.2f} minutes")

In [8]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam

classifier = Sequential([
    Dense(128, activation="relu", input_shape=(2048,)),
    Dropout(0.3),
    Dense(NUM_CLASSES, activation="softmax")
])

classifier.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

classifier.summary()

In [9]:
EPOCHS_CLASSIFIER = 20

classifier_history = classifier.fit(
    train_features,
    train_labels,
    validation_data=(val_features, val_labels),
    epochs=EPOCHS_CLASSIFIER,
    batch_size=16,
    verbose=1
)

In [10]:
print("Final training accuracy:", classifier_history.history["accuracy"][-1])
print("Final validation accuracy:", classifier_history.history["val_accuracy"][-1])

print("Final training loss:", classifier_history.history["loss"][-1])
print("Final validation loss:", classifier_history.history["val_loss"][-1])

best_epoch = np.argmax(classifier_history.history["val_accuracy"]) + 1
best_val_accuracy = max(classifier_history.history["val_accuracy"])

print("Best validation epoch:", best_epoch)
print("Best validation accuracy:", best_val_accuracy)

In [11]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Generate predictions for the test set
test_probabilities = classifier.predict(
    test_features,
    verbose=0
)

test_predictions = np.argmax(test_probabilities, axis=1)

# Calculate metrics
test_accuracy = accuracy_score(test_labels, test_predictions)

test_precision = precision_score(
    test_labels,
    test_predictions,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    test_labels,
    test_predictions,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    test_labels,
    test_predictions,
    average="weighted",
    zero_division=0
)

print("TEST SET RESULTS")
print("----------------")
print(f"Accuracy :  {test_accuracy:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall   :  {test_recall:.4f}")
print(f"F1-score :  {test_f1:.4f}")

print("\nCLASSIFICATION REPORT")
print("---------------------")

print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=class_names,
        zero_division=0
    )
)

print("\nCONFUSION MATRIX")
print("----------------")

cm = confusion_matrix(
    test_labels,
    test_predictions
)

print(cm)

In [12]:
.\venv_ml\Scripts\python.exe -m pip install scikit-learn

In [13]:
import sklearn

print("scikit-learn version:", sklearn.__version__)
print("scikit-learn is working.")

In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Generate predictions for the test set
test_probabilities = classifier.predict(
    test_features,
    verbose=0
)

test_predictions = np.argmax(test_probabilities, axis=1)

# Calculate metrics
test_accuracy = accuracy_score(test_labels, test_predictions)

test_precision = precision_score(
    test_labels,
    test_predictions,
    average="weighted",
    zero_division=0
)

test_recall = recall_score(
    test_labels,
    test_predictions,
    average="weighted",
    zero_division=0
)

test_f1 = f1_score(
    test_labels,
    test_predictions,
    average="weighted",
    zero_division=0
)

print("TEST SET RESULTS")
print("----------------")
print(f"Accuracy :  {test_accuracy:.4f}")
print(f"Precision:  {test_precision:.4f}")
print(f"Recall   :  {test_recall:.4f}")
print(f"F1-score :  {test_f1:.4f}")

print("\nCLASSIFICATION REPORT")
print("---------------------")

print(
    classification_report(
        test_labels,
        test_predictions,
        target_names=class_names,
        zero_division=0
    )
)

print("\nCONFUSION MATRIX")
print("----------------")

cm = confusion_matrix(
    test_labels,
    test_predictions
)

print(cm)

In [15]:
print("Confusion Matrix:")
print(cm)

print("\nClass order:")
for i, name in enumerate(class_names):
    print(i, "=", name)

In [16]:
import os

# Get test image paths in the same class/order used by image_dataset_from_directory
test_image_paths = []

for class_name in class_names:
    class_dir = os.path.join(TEST_DIR, class_name)

    for filename in sorted(os.listdir(class_dir)):
        file_path = os.path.join(class_dir, filename)

        if os.path.isfile(file_path):
            test_image_paths.append(file_path)

# Verify count
print("Total test image paths:", len(test_image_paths))

# Find incorrect predictions
incorrect_indices = np.where(test_labels != test_predictions)[0]

print("Total incorrect predictions:", len(incorrect_indices))
print()

for i in incorrect_indices:
    actual = class_names[test_labels[i]]
    predicted = class_names[test_predictions[i]]
    filename = os.path.basename(test_image_paths[i])

    print(
        f"Index {i:3d} | "
        f"Actual: {actual:15s} | "
        f"Predicted: {predicted:15s} | "
        f"File: {filename}"
    )

In [17]:
import pandas as pd

error_rows = []

for i in incorrect_indices:
    error_rows.append({
        "Index": i,
        "Image": os.path.basename(test_image_paths[i]),
        "Actual": class_names[test_labels[i]],
        "Predicted": class_names[test_predictions[i]],
        "Confidence": float(np.max(test_probabilities[i]))
    })

error_df = pd.DataFrame(error_rows)

print(error_df.to_string(index=False))

In [18]:
import pandas as pd

error_rows = []

for i in incorrect_indices:
    error_rows.append({
        "Index": i,
        "Image": os.path.basename(test_image_paths[i]),
        "Actual": class_names[test_labels[i]],
        "Predicted": class_names[test_predictions[i]],
        "Confidence": float(np.max(test_probabilities[i]))
    })

error_df = pd.DataFrame(error_rows)

print(error_df.to_string(index=False))

In [19]:
import os

MODEL_DIR = os.path.join(PROJECT_DIR, "models", "cnn")

os.makedirs(MODEL_DIR, exist_ok=True)

# Save the trained classifier
classifier.save(
    os.path.join(MODEL_DIR, "resnet50_soil_classifier.keras")
)

# Save the ResNet-50 feature extractor
feature_extractor.save(
    os.path.join(MODEL_DIR, "resnet50_feature_extractor.keras")
)

print("Models saved successfully.")
print("Location:", MODEL_DIR)

In [20]:
print("classifier:", "classifier" in globals())
print("feature_extractor:", "feature_extractor" in globals())
print("train_features:", "train_features" in globals())
print("test_features:", "test_features" in globals())
print("test_predictions:", "test_predictions" in globals())

In [21]:
print("classifier:", "classifier" in globals())
print("feature_extractor:", "feature_extractor" in globals())
print("train_features:", "train_features" in globals())
print("test_features:", "test_features" in globals())
print("test_predictions:", "test_predictions" in globals())

In [22]:
import os

MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models",
    "cnn"
)

os.makedirs(MODEL_DIR, exist_ok=True)

classifier_path = os.path.join(
    MODEL_DIR,
    "resnet50_soil_classifier.keras"
)

classifier.save(classifier_path)

print("Classifier saved successfully.")
print("Saved to:", classifier_path)

In [23]:
import matplotlib.pyplot as plt

# Create output directory
PLOTS_DIR = os.path.join(PROJECT_DIR, "reports", "cnn")
os.makedirs(PLOTS_DIR, exist_ok=True)

# -----------------------------
# Accuracy plot
# -----------------------------
plt.figure(figsize=(8, 5))

plt.plot(
    classifier_history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    classifier_history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ResNet-50 Soil Classification - Accuracy")
plt.legend()
plt.grid(True)

accuracy_plot_path = os.path.join(
    PLOTS_DIR,
    "resnet50_accuracy.png"
)

plt.savefig(accuracy_plot_path, dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# Loss plot
# -----------------------------
plt.figure(figsize=(8, 5))

plt.plot(
    classifier_history.history["loss"],
    label="Training Loss"
)

plt.plot(
    classifier_history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet-50 Soil Classification - Loss")
plt.legend()
plt.grid(True)

loss_plot_path = os.path.join(
    PLOTS_DIR,
    "resnet50_loss.png"
)

plt.savefig(loss_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Plots saved successfully.")
print("Accuracy plot:", accuracy_plot_path)
print("Loss plot:", loss_plot_path)

In [24]:
import matplotlib.pyplot as plt

# Create output directory
PLOTS_DIR = os.path.join(PROJECT_DIR, "reports", "cnn")
os.makedirs(PLOTS_DIR, exist_ok=True)

# -----------------------------
# Accuracy plot
# -----------------------------
plt.figure(figsize=(8, 5))

plt.plot(
    classifier_history.history["accuracy"],
    label="Training Accuracy"
)

plt.plot(
    classifier_history.history["val_accuracy"],
    label="Validation Accuracy"
)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("ResNet-50 Soil Classification - Accuracy")
plt.legend()
plt.grid(True)

accuracy_plot_path = os.path.join(
    PLOTS_DIR,
    "resnet50_accuracy.png"
)

plt.savefig(accuracy_plot_path, dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------
# Loss plot
# -----------------------------
plt.figure(figsize=(8, 5))

plt.plot(
    classifier_history.history["loss"],
    label="Training Loss"
)

plt.plot(
    classifier_history.history["val_loss"],
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("ResNet-50 Soil Classification - Loss")
plt.legend()
plt.grid(True)

loss_plot_path = os.path.join(
    PLOTS_DIR,
    "resnet50_loss.png"
)

plt.savefig(loss_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("Plots saved successfully.")
print("Accuracy plot:", accuracy_plot_path)
print("Loss plot:", loss_plot_path)

In [25]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 7))

plt.imshow(cm, interpolation="nearest")
plt.title("ResNet-50 Soil Classification - Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")

plt.xticks(
    range(len(class_names)),
    class_names,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(class_names)),
    class_names
)

# Add numbers inside the matrix
for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(
            j,
            i,
            cm[i, j],
            ha="center",
            va="center"
        )

plt.colorbar()
plt.tight_layout()

cm_plot_path = os.path.join(
    PLOTS_DIR,
    "resnet50_confusion_matrix.png"
)

plt.savefig(
    cm_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Confusion matrix plot saved successfully.")
print("Saved to:", cm_plot_path)

In [26]:
# ==========================================
# WEEK 3 - CNN SOIL IMAGE ANALYSIS REPORT
# ==========================================

import os

REPORT_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn"
)

os.makedirs(REPORT_DIR, exist_ok=True)

report_path = os.path.join(
    REPORT_DIR,
    "Week_3_CNN_Evaluation_Report.md"
)

# Best validation result
best_epoch = np.argmax(
    classifier_history.history["val_accuracy"]
) + 1

best_val_accuracy = max(
    classifier_history.history["val_accuracy"]
)

# Final training values
final_train_accuracy = classifier_history.history["accuracy"][-1]
final_val_accuracy = classifier_history.history["val_accuracy"][-1]

final_train_loss = classifier_history.history["loss"][-1]
final_val_loss = classifier_history.history["val_loss"][-1]

# Create report
report = f"""# Week 3 — CNN Soil Image Analysis Evaluation Report

## 1. Objective

Develop a CNN-based soil image classification system using transfer learning
with the selected ResNet-50 architecture.

## 2. Dataset

- Training images: 794
- Validation images: 171
- Test images: 174
- Total images: 1,139
- Number of soil classes: 7

### Soil Classes

1. Alluvial Soil
2. Arid Soil
3. Black Soil
4. Laterite Soil
5. Mountain Soil
6. Red Soil
7. Yellow Soil

## 3. CNN Model

### Architecture

- Base model: ResNet-50
- Pretrained weights: ImageNet
- Pretrained layers: Frozen
- Global Average Pooling: Yes
- Dropout: 0.3
- Final classification layer: Dense
- Number of output classes: 7
- Activation: Softmax

### Transfer Learning Configuration

The pretrained ResNet-50 network was used as a frozen feature extractor.
Each image was converted into a 2,048-dimensional feature representation.

A lightweight classifier was trained on these extracted features.

## 4. Training Configuration

- Image size: 224 × 224
- Batch size: 8 for image processing
- Classifier epochs: 20
- Optimizer: Adam
- Learning rate: 0.0001
- Loss function: Sparse Categorical Crossentropy

## 5. Training Results

- Final training accuracy: {final_train_accuracy:.4f} ({final_train_accuracy*100:.2f}%)
- Final validation accuracy: {final_val_accuracy:.4f} ({final_val_accuracy*100:.2f}%)
- Final training loss: {final_train_loss:.4f}
- Final validation loss: {final_val_loss:.4f}
- Best validation epoch: {best_epoch}
- Best validation accuracy: {best_val_accuracy:.4f} ({best_val_accuracy*100:.2f}%)

## 6. Test Set Evaluation

- Test accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)
- Weighted precision: {test_precision:.4f} ({test_precision*100:.2f}%)
- Weighted recall: {test_recall:.4f} ({test_recall*100:.2f}%)
- Weighted F1-score: {test_f1:.4f} ({test_f1*100:.2f}%)

## 7. Class-wise Performance

| Class | Precision | Recall | F1-score | Support |
|---|---:|---:|---:|---:|
"""

# Add class-wise classification metrics
from sklearn.metrics import classification_report

class_report = classification_report(
    test_labels,
    test_predictions,
    target_names=class_names,
    output_dict=True,
    zero_division=0
)

for class_name in class_names:
    metrics = class_report[class_name]

    report += (
        f"| {class_name} | "
        f"{metrics['precision']:.2f} | "
        f"{metrics['recall']:.2f} | "
        f"{metrics['f1-score']:.2f} | "
        f"{int(metrics['support'])} |\n"
    )

report += f"""
## 8. Confusion Matrix

The confusion matrix was generated to identify correct and incorrect
classification patterns across the seven soil classes.

![Confusion Matrix](resnet50_confusion_matrix.png)

## 9. Incorrect Prediction Analysis

Total test images: {len(test_labels)}

Correct predictions: {np.sum(test_labels == test_predictions)}

Incorrect predictions: {len(incorrect_indices)}

The largest classification difficulty was observed for Alluvial Soil,
where 3 of 8 test samples were correctly classified.

Red Soil achieved 17 of 17 correct predictions in the test set.

The detailed incorrect-prediction information, including prediction
confidence, was recorded during the analysis.

## 10. Overfitting / Generalization Observation

Training accuracy reached {final_train_accuracy*100:.2f}%, while validation
accuracy reached {final_val_accuracy*100:.2f}%.

The difference indicates some degree of overfitting. However, the validation
accuracy continued improving through the final training epoch, and the
independent test accuracy was {test_accuracy*100:.2f}%.

## 11. Training Curves

### Accuracy

![Accuracy](resnet50_accuracy.png)

### Loss

![Loss](resnet50_loss.png)

## 12. Model File

The trained classifier was saved as:

`models/cnn/resnet50_soil_classifier.keras`

## 13. Conclusion

The ResNet-50 transfer-learning pipeline successfully classified the seven
soil-image classes and achieved an overall test accuracy of
{test_accuracy*100:.2f}% with a weighted F1-score of {test_f1*100:.2f}%.

The model and evaluation outputs are ready for inclusion in the Week 3
CNN soil image analysis deliverable.
"""

# Save report
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)

print("Week 3 CNN evaluation report created successfully.")
print("Saved to:")
print(report_path)

In [27]:
import os

report_path = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn",
    "Week_3_CNN_Evaluation_Report.md"
)

classifier_path = os.path.join(
    PROJECT_DIR,
    "models",
    "cnn",
    "resnet50_soil_classifier.keras"
)

accuracy_plot = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn",
    "resnet50_accuracy.png"
)

loss_plot = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn",
    "resnet50_loss.png"
)

cm_plot = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn",
    "resnet50_confusion_matrix.png"
)

print("WEEK 3 DELIVERABLE CHECK")
print("-------------------------")
print("Evaluation report :", os.path.exists(report_path))
print("CNN classifier    :", os.path.exists(classifier_path))
print("Accuracy plot     :", os.path.exists(accuracy_plot))
print("Loss plot         :", os.path.exists(loss_plot))
print("Confusion matrix  :", os.path.exists(cm_plot))

In [28]:
import pandas as pd
import os

STRUCTURED_DATA_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "soil_tests",
    "processed",
    "clean_soil_test_data.csv"
)

soil_df = pd.read_csv(STRUCTURED_DATA_PATH)

print("Dataset shape:", soil_df.shape)
print("\nColumns:")
for column in soil_df.columns:
    print("-", column)

print("\nData types:")
print(soil_df.dtypes)

print("\nMissing values:")
print(soil_df.isnull().sum())

In [29]:
print("Nutrient summary:")
print(
    soil_df[
        [
            "nitrogen",
            "phosphorus",
            "potassium",
            "ph",
            "moisture",
            "organic_matter"
        ]
    ].describe()
)

print("\nCrop values:")
print(soil_df["crop"].value_counts(dropna=False))

In [30]:
import pandas as pd
import os

RAW_DATA_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "soil_tests",
    "raw",
    "Samples EJP Probefield.xlsx"
)

raw_df = pd.read_excel(RAW_DATA_PATH)

print("Raw dataset shape:", raw_df.shape)

print("\nAll raw columns:")
for i, column in enumerate(raw_df.columns):
    print(i, ":", repr(column))

In [31]:
import sys
get_ipython().system('{sys.executable} -m pip install openpyxl')

In [32]:
import pandas as pd
import os

RAW_DATA_PATH = os.path.join(
    PROJECT_DIR,
    "data",
    "soil_tests",
    "raw",
    "Samples EJP Probefield.xlsx"
)

raw_df = pd.read_excel(RAW_DATA_PATH)

print("Raw dataset shape:", raw_df.shape)

print("\nAll raw columns:")
for i, column in enumerate(raw_df.columns):
    print(i, ":", repr(column))

In [33]:
# Find columns related to nutrients, deficiency, fertility, and soil parameters

keywords = [
    "nitrogen", "nitrogen", "N ",
    "phosphorus", "phosph", "P ",
    "potassium", "potass", "K ",
    "deficien", "nutrient", "fertility",
    "organic", "moisture", "pH", "carbon"
]

matching_columns = []

for column in raw_df.columns:
    column_text = str(column).lower()

    if any(keyword.lower() in column_text for keyword in keywords):
        matching_columns.append(column)

print("Number of matching columns:", len(matching_columns))

print("\nMatching columns:")
for i, column in enumerate(matching_columns):
    print(i, ":", repr(column))

In [34]:
# Inspect the structure of the raw Excel dataset

print("FIRST 40 COLUMNS")
print("=" * 80)

for i, column in enumerate(raw_df.columns[:40]):
    print(f"{i:3} : {column!r}")


print("\n\nLAST 40 COLUMNS")
print("=" * 80)

start = max(0, len(raw_df.columns) - 40)

for i, column in enumerate(raw_df.columns[start:], start=start):
    print(f"{i:4} : {column!r}")


print("\n\nFIRST 5 ROWS OF IMPORTANT SOIL PARAMETERS")
print("=" * 80)

important_columns = [
    "NUMBER",
    "CROP",
    "Moisture (%dw)",
    "pH (Ext 1:2.5)",
    "organic Matter (%)",
    "total-N (%)",
    "avail-P (mg/kg)",
    "avail-K (mg/kg)"
]

print(raw_df[important_columns].head())

In [35]:
print("Label column information")
print("=" * 60)

print("Data type:", raw_df["Label"].dtype)
print("Missing values:", raw_df["Label"].isna().sum())
print("Unique values:", raw_df["Label"].nunique())

print("\nLabel values:")
print(raw_df["Label"].value_counts(dropna=False))

In [36]:
# WEEK 4 — Step 2: Identify possible structured ML targets

print("=" * 70)
print("AVAILABLE STRUCTURED DATA")
print("=" * 70)

print("\nNumerical soil parameters:")
numeric_features = [
    "moisture",
    "ph",
    "organic_matter",
    "nitrogen",
    "phosphorus",
    "potassium"
]

for feature in numeric_features:
    print(f"  ✓ {feature}")

print("\nCategorical / metadata columns:")

for column in soil_df.columns:
    if soil_df[column].dtype == "object":
        print(f"  • {column}")
        print(f"    Unique values: {soil_df[column].nunique()}")
        print(f"    Missing: {soil_df[column].isna().sum()}")

print("\n" + "=" * 70)
print("CROP VALUE COUNTS")
print("=" * 70)

print(soil_df["crop"].value_counts(dropna=False))

In [37]:
import sys

get_ipython().system('{sys.executable} -m pip install xgboost shap')

In [38]:
# WEEK 4 — Step 4: Prepare XGBoost training/testing datasets

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

# ---------------------------------------------------------
# Required soil parameters
# ---------------------------------------------------------

features = [
    "moisture",
    "ph",
    "organic_matter",
    "nitrogen",
    "phosphorus",
    "potassium"
]

print("Available features:")
for feature in features:
    print("✓", feature)

print("\nDataset shape:", soil_df.shape)

# ---------------------------------------------------------
# Create a clean numerical dataset
# ---------------------------------------------------------

ml_df = soil_df[features].copy()

# Ensure numeric values
for column in features:
    ml_df[column] = pd.to_numeric(ml_df[column], errors="coerce")

# Remove rows with missing values
ml_df = ml_df.dropna().reset_index(drop=True)

print("\nML dataset shape:", ml_df.shape)
print("Missing values:")
print(ml_df.isna().sum())

# ---------------------------------------------------------
# Create separate datasets for N, P and K prediction
# ---------------------------------------------------------

# Nitrogen prediction
X_N = ml_df[
    ["moisture", "ph", "organic_matter", "phosphorus", "potassium"]
]
y_N = ml_df["nitrogen"]

# Phosphorus prediction
X_P = ml_df[
    ["moisture", "ph", "organic_matter", "nitrogen", "potassium"]
]
y_P = ml_df["phosphorus"]

# Potassium prediction
X_K = ml_df[
    ["moisture", "ph", "organic_matter", "nitrogen", "phosphorus"]
]
y_K = ml_df["potassium"]

# ---------------------------------------------------------
# Use the SAME train/test rows for all three targets
# ---------------------------------------------------------

indices = np.arange(len(ml_df))

train_idx, test_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42
)

X_N_train = X_N.iloc[train_idx]
X_N_test  = X_N.iloc[test_idx]
y_N_train = y_N.iloc[train_idx]
y_N_test  = y_N.iloc[test_idx]

X_P_train = X_P.iloc[train_idx]
X_P_test  = X_P.iloc[test_idx]
y_P_train = y_P.iloc[train_idx]
y_P_test  = y_P.iloc[test_idx]

X_K_train = X_K.iloc[train_idx]
X_K_test  = X_K.iloc[test_idx]
y_K_train = y_K.iloc[train_idx]
y_K_test  = y_K.iloc[test_idx]

# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("TRAIN / TEST SPLIT")
print("=" * 70)

print("Training samples:", len(train_idx))
print("Testing samples :", len(test_idx))

print("\nNitrogen:")
print("X:", X_N_train.shape, "→", X_N_test.shape)
print("y:", y_N_train.shape, "→", y_N_test.shape)

print("\nPhosphorus:")
print("X:", X_P_train.shape, "→", X_P_test.shape)
print("y:", y_P_train.shape, "→", y_P_test.shape)

print("\nPotassium:")
print("X:", X_K_train.shape, "→", X_K_test.shape)
print("y:", y_K_train.shape, "→", y_K_test.shape)

print("\n" + "=" * 70)
print("FEATURES USED FOR EACH MODEL")
print("=" * 70)

print("Nitrogen target    :", list(X_N.columns))
print("Phosphorus target  :", list(X_P.columns))
print("Potassium target   :", list(X_K.columns))

In [39]:
# WEEK 4 — Step 5: Train XGBoost models

from xgboost import XGBRegressor

print("=" * 70)
print("TRAINING XGBOOST MODELS")
print("=" * 70)

# ---------------------------------------------------------
# XGBoost model configuration
# ---------------------------------------------------------

xgb_params = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.05,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "reg:squarederror",
    "random_state": 42,
    "n_jobs": -1
}

# ---------------------------------------------------------
# Nitrogen model
# ---------------------------------------------------------

print("\nTraining Nitrogen model...")

xgb_N = XGBRegressor(**xgb_params)

xgb_N.fit(
    X_N_train,
    y_N_train
)

print("✓ Nitrogen model trained")


# ---------------------------------------------------------
# Phosphorus model
# ---------------------------------------------------------

print("\nTraining Phosphorus model...")

xgb_P = XGBRegressor(**xgb_params)

xgb_P.fit(
    X_P_train,
    y_P_train
)

print("✓ Phosphorus model trained")


# ---------------------------------------------------------
# Potassium model
# ---------------------------------------------------------

print("\nTraining Potassium model...")

xgb_K = XGBRegressor(**xgb_params)

xgb_K.fit(
    X_K_train,
    y_K_train
)

print("✓ Potassium model trained")


print("\n" + "=" * 70)
print("ALL XGBOOST MODELS TRAINED SUCCESSFULLY")
print("=" * 70)

In [40]:
# WEEK 4 — Step 6: Evaluate XGBoost models

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=" * 70)
print("XGBOOST MODEL EVALUATION")
print("=" * 70)

# ---------------------------------------------------------
# Generate predictions
# ---------------------------------------------------------

y_N_pred = xgb_N.predict(X_N_test)
y_P_pred = xgb_P.predict(X_P_test)
y_K_pred = xgb_K.predict(X_K_test)


# ---------------------------------------------------------
# Evaluation function
# ---------------------------------------------------------

def evaluate_regression_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    print(f"\n{name}")
    print("-" * 50)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


# ---------------------------------------------------------
# Evaluate all three models
# ---------------------------------------------------------

results_N = evaluate_regression_model(
    "XGBoost — Nitrogen",
    y_N_test,
    y_N_pred
)

results_P = evaluate_regression_model(
    "XGBoost — Phosphorus",
    y_P_test,
    y_P_pred
)

results_K = evaluate_regression_model(
    "XGBoost — Potassium",
    y_K_test,
    y_K_pred
)


# ---------------------------------------------------------
# Combined results table
# ---------------------------------------------------------

xgb_results = pd.DataFrame([
    results_N,
    results_P,
    results_K
])

print("\n" + "=" * 70)
print("FINAL XGBOOST RESULTS")
print("=" * 70)

print(xgb_results.to_string(index=False))

In [41]:
# WEEK 4 — Step 7: XGBoost Hyperparameter Tuning

from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV

print("=" * 70)
print("XGBOOST HYPERPARAMETER TUNING")
print("=" * 70)

# ---------------------------------------------------------
# Compact parameter grid
# Suitable for our small dataset
# ---------------------------------------------------------

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [2, 3],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.8],
    "colsample_bytree": [0.8]
}

# ---------------------------------------------------------
# Function for tuning one target
# ---------------------------------------------------------

def tune_xgboost(X_train, y_train, target_name):

    print(f"\nTuning {target_name} model...")

    model = XGBRegressor(
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    grid_search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="neg_root_mean_squared_error",
        cv=5,
        n_jobs=-1,
        refit=True
    )

    grid_search.fit(X_train, y_train)

    print("✓ Tuning completed")
    print("Best parameters:")
    print(grid_search.best_params_)
    print(f"Best CV RMSE: {-grid_search.best_score_:.4f}")

    return grid_search.best_estimator_


# ---------------------------------------------------------
# Tune N, P and K models
# ---------------------------------------------------------

best_xgb_N = tune_xgboost(
    X_N_train,
    y_N_train,
    "Nitrogen"
)

best_xgb_P = tune_xgboost(
    X_P_train,
    y_P_train,
    "Phosphorus"
)

best_xgb_K = tune_xgboost(
    X_K_train,
    y_K_train,
    "Potassium"
)

print("\n" + "=" * 70)
print("HYPERPARAMETER TUNING COMPLETED")
print("=" * 70)

In [42]:
# WEEK 4 — Step 8: Evaluate Tuned XGBoost Models

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("=" * 70)
print("TUNED XGBOOST TEST-SET EVALUATION")
print("=" * 70)


def evaluate_tuned_model(name, model, X_test, y_test):
    predictions = model.predict(X_test)

    mae = mean_absolute_error(y_test, predictions)
    rmse = np.sqrt(mean_squared_error(y_test, predictions))
    r2 = r2_score(y_test, predictions)

    print(f"\n{name}")
    print("-" * 50)
    print(f"MAE  : {mae:.4f}")
    print(f"RMSE : {rmse:.4f}")
    print(f"R²   : {r2:.4f}")

    return predictions, {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    }


# Nitrogen
y_N_tuned_pred, tuned_N_results = evaluate_tuned_model(
    "Tuned XGBoost — Nitrogen",
    best_xgb_N,
    X_N_test,
    y_N_test
)

# Phosphorus
y_P_tuned_pred, tuned_P_results = evaluate_tuned_model(
    "Tuned XGBoost — Phosphorus",
    best_xgb_P,
    X_P_test,
    y_P_test
)

# Potassium
y_K_tuned_pred, tuned_K_results = evaluate_tuned_model(
    "Tuned XGBoost — Potassium",
    best_xgb_K,
    X_K_test,
    y_K_test
)


# ---------------------------------------------------------
# Final comparison table
# ---------------------------------------------------------

tuned_xgb_results = pd.DataFrame([
    tuned_N_results,
    tuned_P_results,
    tuned_K_results
])

print("\n" + "=" * 70)
print("FINAL TUNED XGBOOST RESULTS")
print("=" * 70)

print(tuned_xgb_results.to_string(index=False))

In [43]:
# WEEK 4 — Step 9: Save final tuned XGBoost models

import os

XGB_MODEL_DIR = os.path.join(
    PROJECT_DIR,
    "models",
    "xgboost"
)

os.makedirs(XGB_MODEL_DIR, exist_ok=True)

# Save models
N_MODEL_PATH = os.path.join(
    XGB_MODEL_DIR,
    "xgboost_nitrogen_final.json"
)

P_MODEL_PATH = os.path.join(
    XGB_MODEL_DIR,
    "xgboost_phosphorus_final.json"
)

K_MODEL_PATH = os.path.join(
    XGB_MODEL_DIR,
    "xgboost_potassium_final.json"
)

best_xgb_N.save_model(N_MODEL_PATH)
best_xgb_P.save_model(P_MODEL_PATH)
best_xgb_K.save_model(K_MODEL_PATH)

print("=" * 70)
print("FINAL XGBOOST MODELS SAVED")
print("=" * 70)

print("\nNitrogen   :", N_MODEL_PATH)
print("Phosphorus :", P_MODEL_PATH)
print("Potassium  :", K_MODEL_PATH)

print("\nVerification:")
print("Nitrogen   :", os.path.exists(N_MODEL_PATH))
print("Phosphorus :", os.path.exists(P_MODEL_PATH))
print("Potassium  :", os.path.exists(K_MODEL_PATH))

In [44]:
# WEEK 4 — Step 10: SHAP analysis for XGBoost Nitrogen model

import os
import shap
import matplotlib.pyplot as plt

print("=" * 70)
print("SHAP ANALYSIS — NITROGEN MODEL")
print("=" * 70)

# Create SHAP explainer
explainer_N = shap.TreeExplainer(best_xgb_N)

# Calculate SHAP values for the test set
shap_values_N = explainer_N.shap_values(X_N_test)

print("✓ SHAP values calculated")

print("\nSHAP values shape:", shap_values_N.shape)
print("Test data shape  :", X_N_test.shape)

# ---------------------------------------------------------
# SHAP feature importance
# ---------------------------------------------------------

shap_importance_N = pd.DataFrame({
    "Feature": X_N_test.columns,
    "Mean_Absolute_SHAP": np.abs(shap_values_N).mean(axis=0)
})

shap_importance_N = shap_importance_N.sort_values(
    "Mean_Absolute_SHAP",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 70)
print("NITROGEN — SHAP FEATURE IMPORTANCE")
print("=" * 70)

print(shap_importance_N.to_string(index=False))

# ---------------------------------------------------------
# SHAP summary plot
# ---------------------------------------------------------

plt.figure()

shap.summary_plot(
    shap_values_N,
    X_N_test,
    show=False
)

plt.title("SHAP Feature Importance — Nitrogen XGBoost")

plt.tight_layout()

SHAP_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "xgboost",
    "shap"
)

os.makedirs(SHAP_DIR, exist_ok=True)

SHAP_N_PATH = os.path.join(
    SHAP_DIR,
    "shap_nitrogen_summary.png"
)

plt.savefig(
    SHAP_N_PATH,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("\n✓ SHAP plot saved:")
print(SHAP_N_PATH)
print("\nFile exists:", os.path.exists(SHAP_N_PATH))

In [45]:
# WEEK 4 — Step 11: SHAP analysis for Phosphorus and Potassium

import os
import shap
import matplotlib.pyplot as plt

SHAP_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "xgboost",
    "shap"
)

os.makedirs(SHAP_DIR, exist_ok=True)


def create_shap_analysis(model, X_test, target_name, file_name):
    print("\n" + "=" * 70)
    print(f"SHAP ANALYSIS — {target_name.upper()} MODEL")
    print("=" * 70)

    # Create SHAP explainer
    explainer = shap.TreeExplainer(model)

    # Calculate SHAP values
    shap_values = explainer.shap_values(X_test)

    print("✓ SHAP values calculated")
    print("SHAP values shape:", shap_values.shape)

    # Feature importance
    importance = pd.DataFrame({
        "Feature": X_test.columns,
        "Mean_Absolute_SHAP": np.abs(shap_values).mean(axis=0)
    })

    importance = importance.sort_values(
        "Mean_Absolute_SHAP",
        ascending=False
    ).reset_index(drop=True)

    print("\nFeature importance:")
    print(importance.to_string(index=False))

    # Summary plot
    plt.figure()

    shap.summary_plot(
        shap_values,
        X_test,
        show=False
    )

    plt.title(f"SHAP Feature Importance — {target_name} XGBoost")

    plt.tight_layout()

    output_path = os.path.join(
        SHAP_DIR,
        file_name
    )

    plt.savefig(
        output_path,
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    print("\n✓ Plot saved:")
    print(output_path)
    print("File exists:", os.path.exists(output_path))

    return explainer, shap_values, importance


# ---------------------------------------------------------
# Phosphorus
# ---------------------------------------------------------

explainer_P, shap_values_P, shap_importance_P = create_shap_analysis(
    best_xgb_P,
    X_P_test,
    "Phosphorus",
    "shap_phosphorus_summary.png"
)


# ---------------------------------------------------------
# Potassium
# ---------------------------------------------------------

explainer_K, shap_values_K, shap_importance_K = create_shap_analysis(
    best_xgb_K,
    X_K_test,
    "Potassium",
    "shap_potassium_summary.png"
)


print("\n" + "=" * 70)
print("SHAP ANALYSIS COMPLETED")
print("=" * 70)

In [46]:
# WEEK 4 — Step 12: Reconstruct ResNet-50 for Grad-CAM

import os
import numpy as np
import tensorflow as tf

from tensorflow.keras.applications import ResNet50

print("=" * 70)
print("RECONSTRUCTING RESNET-50 FOR GRAD-CAM")
print("=" * 70)

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

CNN_MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "models",
    "cnn",
    "resnet50_soil_classifier.keras"
)

print("Saved classifier:")
print(CNN_MODEL_PATH)
print("Exists:", os.path.exists(CNN_MODEL_PATH))

# ---------------------------------------------------------
# Load the saved classifier
# ---------------------------------------------------------

classifier = tf.keras.models.load_model(
    CNN_MODEL_PATH
)

print("\n✓ Saved classifier loaded")
print("Classifier input shape :", classifier.input_shape)
print("Classifier output shape:", classifier.output_shape)

# ---------------------------------------------------------
# Recreate frozen ResNet-50 feature extractor
# ---------------------------------------------------------

resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)

resnet_base.trainable = False

print("\n✓ ResNet-50 feature extractor loaded")
print("Feature output shape:", resnet_base.output_shape)

# ---------------------------------------------------------
# Build combined model
# ---------------------------------------------------------

inputs = tf.keras.Input(
    shape=(224, 224, 3),
    name="soil_image"
)

features = resnet_base(inputs)

outputs = classifier(features)

gradcam_model = tf.keras.Model(
    inputs=inputs,
    outputs=outputs,
    name="resnet50_gradcam_model"
)

print("\n✓ Combined Grad-CAM model created")
print("Input :", gradcam_model.input_shape)
print("Output:", gradcam_model.output_shape)

print("\n" + "=" * 70)
print("GRAD-CAM MODEL READY")
print("=" * 70)

In [47]:
# WEEK 4 — Step 13: Generate Grad-CAM heatmap

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing import image

print("=" * 70)
print("GRAD-CAM — SOIL IMAGE EXPLANATION")
print("=" * 70)

# ---------------------------------------------------------
# Find the first test image
# ---------------------------------------------------------

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "images",
    "split",
    "test"
)

class_names = [
    "Alluvial_Soil",
    "Arid_Soil",
    "Black_Soil",
    "Laterite_Soil",
    "Mountain_Soil",
    "Red_Soil",
    "Yellow_Soil"
]

image_path = None
actual_class = None

for class_name in class_names:
    class_dir = os.path.join(TEST_DIR, class_name)

    if os.path.exists(class_dir):
        files = sorted([
            f for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        if files:
            image_path = os.path.join(class_dir, files[0])
            actual_class = class_name
            break

if image_path is None:
    raise FileNotFoundError("No test image was found.")

print("Image:", image_path)
print("Actual class:", actual_class)


# ---------------------------------------------------------
# Load and preprocess image
# ---------------------------------------------------------

img = image.load_img(
    image_path,
    target_size=(224, 224)
)

img_array = image.img_to_array(img)

input_array = np.expand_dims(
    img_array,
    axis=0
)

# ResNet50 preprocessing
input_array = tf.keras.applications.resnet50.preprocess_input(
    input_array
)


# ---------------------------------------------------------
# Prediction
# ---------------------------------------------------------

predictions = gradcam_model.predict(
    input_array,
    verbose=0
)

predicted_index = np.argmax(predictions[0])

predicted_class = class_names[predicted_index]
confidence = float(predictions[0][predicted_index])

print("\nPrediction:")
print("Predicted class:", predicted_class)
print(f"Confidence: {confidence:.4f}")


# ---------------------------------------------------------
# Grad-CAM
# ---------------------------------------------------------

# Get the last convolutional layer of ResNet-50
last_conv_layer = None

for layer in reversed(resnet_base.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer = layer
        break

if last_conv_layer is None:
    raise ValueError("Could not find a convolutional layer.")

print("\nGrad-CAM layer:", last_conv_layer.name)


# Create model that returns:
# 1. convolutional feature maps
# 2. final prediction

grad_model = tf.keras.Model(
    inputs=gradcam_model.inputs,
    outputs=[
        last_conv_layer.output,
        gradcam_model.output
    ]
)


# Calculate gradients
with tf.GradientTape() as tape:

    conv_outputs, predictions_tensor = grad_model(
        input_array
    )

    class_score = predictions_tensor[:, predicted_index]


gradients = tape.gradient(
    class_score,
    conv_outputs
)


# Global average pooling of gradients
pooled_gradients = tf.reduce_mean(
    gradients,
    axis=(0, 1, 2)
)


# Remove batch dimension
conv_outputs = conv_outputs[0]


# Weight feature maps
heatmap = tf.reduce_sum(
    conv_outputs * pooled_gradients,
    axis=-1
)


# ReLU
heatmap = tf.maximum(
    heatmap,
    0
)


# Normalize
max_value = tf.reduce_max(heatmap)

if max_value > 0:
    heatmap = heatmap / max_value

heatmap = heatmap.numpy()


# ---------------------------------------------------------
# Resize heatmap to original image size
# ---------------------------------------------------------

heatmap_resized = tf.image.resize(
    heatmap[..., np.newaxis],
    (224, 224)
).numpy().squeeze()


# ---------------------------------------------------------
# Display Grad-CAM
# ---------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title(
    f"Original\nActual: {actual_class}"
)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img)
plt.imshow(
    heatmap_resized,
    alpha=0.45
)
plt.title(
    f"Grad-CAM\nPredicted: {predicted_class} ({confidence:.2%})"
)
plt.axis("off")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# Save Grad-CAM image
# ---------------------------------------------------------

GRADCAM_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn",
    "gradcam"
)

os.makedirs(
    GRADCAM_DIR,
    exist_ok=True
)

GRADCAM_PATH = os.path.join(
    GRADCAM_DIR,
    "gradcam_first_test_image.png"
)

plt.savefig(
    GRADCAM_PATH,
    dpi=300,
    bbox_inches="tight"
)

print("\n✓ Grad-CAM saved:")
print(GRADCAM_PATH)

print("\nFile exists:", os.path.exists(GRADCAM_PATH))

In [48]:
import sys

get_ipython().system('{sys.executable} -m pip install pillow')

In [49]:
from PIL import Image

print("Pillow installed successfully")
print("Version:", Image.__version__)

In [50]:
# WEEK 4 — Step 13: Generate Grad-CAM heatmap

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing import image

print("=" * 70)
print("GRAD-CAM — SOIL IMAGE EXPLANATION")
print("=" * 70)

# ---------------------------------------------------------
# Find the first test image
# ---------------------------------------------------------

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "images",
    "split",
    "test"
)

class_names = [
    "Alluvial_Soil",
    "Arid_Soil",
    "Black_Soil",
    "Laterite_Soil",
    "Mountain_Soil",
    "Red_Soil",
    "Yellow_Soil"
]

image_path = None
actual_class = None

for class_name in class_names:
    class_dir = os.path.join(TEST_DIR, class_name)

    if os.path.exists(class_dir):
        files = sorted([
            f for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        if files:
            image_path = os.path.join(class_dir, files[0])
            actual_class = class_name
            break

if image_path is None:
    raise FileNotFoundError("No test image was found.")

print("Image:", image_path)
print("Actual class:", actual_class)


# ---------------------------------------------------------
# Load and preprocess image
# ---------------------------------------------------------

img = image.load_img(
    image_path,
    target_size=(224, 224)
)

img_array = image.img_to_array(img)

input_array = np.expand_dims(
    img_array,
    axis=0
)

# ResNet50 preprocessing
input_array = tf.keras.applications.resnet50.preprocess_input(
    input_array
)


# ---------------------------------------------------------
# Prediction
# ---------------------------------------------------------

predictions = gradcam_model.predict(
    input_array,
    verbose=0
)

predicted_index = np.argmax(predictions[0])

predicted_class = class_names[predicted_index]
confidence = float(predictions[0][predicted_index])

print("\nPrediction:")
print("Predicted class:", predicted_class)
print(f"Confidence: {confidence:.4f}")


# ---------------------------------------------------------
# Grad-CAM
# ---------------------------------------------------------

# Get the last convolutional layer of ResNet-50
last_conv_layer = None

for layer in reversed(resnet_base.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer = layer
        break

if last_conv_layer is None:
    raise ValueError("Could not find a convolutional layer.")

print("\nGrad-CAM layer:", last_conv_layer.name)


# Create model that returns:
# 1. convolutional feature maps
# 2. final prediction

grad_model = tf.keras.Model(
    inputs=gradcam_model.inputs,
    outputs=[
        last_conv_layer.output,
        gradcam_model.output
    ]
)


# Calculate gradients
with tf.GradientTape() as tape:

    conv_outputs, predictions_tensor = grad_model(
        input_array
    )

    class_score = predictions_tensor[:, predicted_index]


gradients = tape.gradient(
    class_score,
    conv_outputs
)


# Global average pooling of gradients
pooled_gradients = tf.reduce_mean(
    gradients,
    axis=(0, 1, 2)
)


# Remove batch dimension
conv_outputs = conv_outputs[0]


# Weight feature maps
heatmap = tf.reduce_sum(
    conv_outputs * pooled_gradients,
    axis=-1
)


# ReLU
heatmap = tf.maximum(
    heatmap,
    0
)


# Normalize
max_value = tf.reduce_max(heatmap)

if max_value > 0:
    heatmap = heatmap / max_value

heatmap = heatmap.numpy()


# ---------------------------------------------------------
# Resize heatmap to original image size
# ---------------------------------------------------------

heatmap_resized = tf.image.resize(
    heatmap[..., np.newaxis],
    (224, 224)
).numpy().squeeze()


# ---------------------------------------------------------
# Display Grad-CAM
# ---------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title(
    f"Original\nActual: {actual_class}"
)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img)
plt.imshow(
    heatmap_resized,
    alpha=0.45
)
plt.title(
    f"Grad-CAM\nPredicted: {predicted_class} ({confidence:.2%})"
)
plt.axis("off")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# Save Grad-CAM image
# ---------------------------------------------------------

GRADCAM_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn",
    "gradcam"
)

os.makedirs(
    GRADCAM_DIR,
    exist_ok=True
)

GRADCAM_PATH = os.path.join(
    GRADCAM_DIR,
    "gradcam_first_test_image.png"
)

plt.savefig(
    GRADCAM_PATH,
    dpi=300,
    bbox_inches="tight"
)

print("\n✓ Grad-CAM saved:")
print(GRADCAM_PATH)

print("\nFile exists:", os.path.exists(GRADCAM_PATH))

In [51]:
# WEEK 4 — Step 13: Generate Grad-CAM heatmap

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.preprocessing import image

print("=" * 70)
print("GRAD-CAM — SOIL IMAGE EXPLANATION")
print("=" * 70)

# ---------------------------------------------------------
# Find the first test image
# ---------------------------------------------------------

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "images",
    "split",
    "test"
)

class_names = [
    "Alluvial_Soil",
    "Arid_Soil",
    "Black_Soil",
    "Laterite_Soil",
    "Mountain_Soil",
    "Red_Soil",
    "Yellow_Soil"
]

image_path = None
actual_class = None

for class_name in class_names:
    class_dir = os.path.join(TEST_DIR, class_name)

    if os.path.exists(class_dir):
        files = sorted([
            f for f in os.listdir(class_dir)
            if f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])

        if files:
            image_path = os.path.join(class_dir, files[0])
            actual_class = class_name
            break

if image_path is None:
    raise FileNotFoundError("No test image was found.")

print("Image:", image_path)
print("Actual class:", actual_class)


# ---------------------------------------------------------
# Load and preprocess image
# ---------------------------------------------------------

img = image.load_img(
    image_path,
    target_size=(224, 224)
)

img_array = image.img_to_array(img)

input_array = np.expand_dims(
    img_array,
    axis=0
)

# ResNet50 preprocessing
input_array = tf.keras.applications.resnet50.preprocess_input(
    input_array
)


# ---------------------------------------------------------
# Prediction
# ---------------------------------------------------------

predictions = gradcam_model.predict(
    input_array,
    verbose=0
)

predicted_index = np.argmax(predictions[0])

predicted_class = class_names[predicted_index]
confidence = float(predictions[0][predicted_index])

print("\nPrediction:")
print("Predicted class:", predicted_class)
print(f"Confidence: {confidence:.4f}")


# ---------------------------------------------------------
# Grad-CAM
# ---------------------------------------------------------

# Get the last convolutional layer of ResNet-50
last_conv_layer = None

for layer in reversed(resnet_base.layers):
    if isinstance(layer, tf.keras.layers.Conv2D):
        last_conv_layer = layer
        break

if last_conv_layer is None:
    raise ValueError("Could not find a convolutional layer.")

print("\nGrad-CAM layer:", last_conv_layer.name)


# Create model that returns:
# 1. convolutional feature maps
# 2. final prediction

grad_model = tf.keras.Model(
    inputs=gradcam_model.inputs,
    outputs=[
        last_conv_layer.output,
        gradcam_model.output
    ]
)


# Calculate gradients
with tf.GradientTape() as tape:

    conv_outputs, predictions_tensor = grad_model(
        input_array
    )

    class_score = predictions_tensor[:, predicted_index]


gradients = tape.gradient(
    class_score,
    conv_outputs
)


# Global average pooling of gradients
pooled_gradients = tf.reduce_mean(
    gradients,
    axis=(0, 1, 2)
)


# Remove batch dimension
conv_outputs = conv_outputs[0]


# Weight feature maps
heatmap = tf.reduce_sum(
    conv_outputs * pooled_gradients,
    axis=-1
)


# ReLU
heatmap = tf.maximum(
    heatmap,
    0
)


# Normalize
max_value = tf.reduce_max(heatmap)

if max_value > 0:
    heatmap = heatmap / max_value

heatmap = heatmap.numpy()


# ---------------------------------------------------------
# Resize heatmap to original image size
# ---------------------------------------------------------

heatmap_resized = tf.image.resize(
    heatmap[..., np.newaxis],
    (224, 224)
).numpy().squeeze()


# ---------------------------------------------------------
# Display Grad-CAM
# ---------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.imshow(img)
plt.title(
    f"Original\nActual: {actual_class}"
)
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(img)
plt.imshow(
    heatmap_resized,
    alpha=0.45
)
plt.title(
    f"Grad-CAM\nPredicted: {predicted_class} ({confidence:.2%})"
)
plt.axis("off")

plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# Save Grad-CAM image
# ---------------------------------------------------------

GRADCAM_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn",
    "gradcam"
)

os.makedirs(
    GRADCAM_DIR,
    exist_ok=True
)

GRADCAM_PATH = os.path.join(
    GRADCAM_DIR,
    "gradcam_first_test_image.png"
)

plt.savefig(
    GRADCAM_PATH,
    dpi=300,
    bbox_inches="tight"
)

print("\n✓ Grad-CAM saved:")
print(GRADCAM_PATH)

print("\nFile exists:", os.path.exists(GRADCAM_PATH))

In [52]:
import sys
import site

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

print("\nSite packages:")
print(site.getsitepackages())

print("\nChecking PIL...")

try:
    import PIL
    from PIL import Image
    print("✓ PIL imported successfully")
    print("Pillow version:", PIL.__version__)
except Exception as e:
    print("✗ PIL import failed")
    print(type(e).__name__, ":", e)

In [53]:
# WEEK 4 — Step 13: Grad-CAM using Pillow directly

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from PIL import Image

print("=" * 70)
print("GRAD-CAM — SOIL IMAGE EXPLANATION")
print("=" * 70)

# ---------------------------------------------------------
# Test image directory
# ---------------------------------------------------------

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "images",
    "split",
    "test"
)

class_names = [
    "Alluvial_Soil",
    "Arid_Soil",
    "Black_Soil",
    "Laterite_Soil",
    "Mountain_Soil",
    "Red_Soil",
    "Yellow_Soil"
]

# ---------------------------------------------------------
# Find first test image
# ---------------------------------------------------------

image_path = None
actual_class = None

for class_name in class_names:

    class_dir = os.path.join(
        TEST_DIR,
        class_name
    )

    if os.path.exists(class_dir):

        files = sorted([
            f for f in os.listdir(class_dir)
            if f.lower().endswith(
                (".jpg", ".jpeg", ".png")
            )
        ])

        if files:
            image_path = os.path.join(
                class_dir,
                files[0]
            )

            actual_class = class_name
            break

if image_path is None:
    raise FileNotFoundError(
        "No test image was found."
    )

print("Image:", image_path)
print("Actual class:", actual_class)


# ---------------------------------------------------------
# Load image using Pillow
# ---------------------------------------------------------

original_image = Image.open(
    image_path
).convert("RGB")

original_image = original_image.resize(
    (224, 224)
)

img_array = np.array(
    original_image,
    dtype=np.float32
)

input_array = np.expand_dims(
    img_array,
    axis=0
)

# ResNet-50 preprocessing
input_array = tf.keras.applications.resnet50.preprocess_input(
    input_array
)

print("✓ Image loaded and preprocessed")


# ---------------------------------------------------------
# Prediction
# ---------------------------------------------------------

predictions = gradcam_model.predict(
    input_array,
    verbose=0
)

predicted_index = int(
    np.argmax(predictions[0])
)

predicted_class = class_names[
    predicted_index
]

confidence = float(
    predictions[0][predicted_index]
)

print("\nPrediction:")
print("Predicted class:", predicted_class)
print(f"Confidence: {confidence:.4f}")


# ---------------------------------------------------------
# Find last convolutional layer
# ---------------------------------------------------------

last_conv_layer = None

for layer in reversed(resnet_base.layers):

    if isinstance(
        layer,
        tf.keras.layers.Conv2D
    ):
        last_conv_layer = layer
        break

if last_conv_layer is None:
    raise ValueError(
        "Could not find convolutional layer."
    )

print("\nGrad-CAM layer:", last_conv_layer.name)


# ---------------------------------------------------------
# Create Grad-CAM model
# ---------------------------------------------------------

grad_model = tf.keras.Model(
    inputs=gradcam_model.inputs,
    outputs=[
        last_conv_layer.output,
        gradcam_model.output
    ]
)


# ---------------------------------------------------------
# Calculate gradients
# ---------------------------------------------------------

with tf.GradientTape() as tape:

    conv_outputs, predictions_tensor = grad_model(
        input_array
    )

    class_score = predictions_tensor[
        :,
        predicted_index
    ]

gradients = tape.gradient(
    class_score,
    conv_outputs
)


# ---------------------------------------------------------
# Calculate Grad-CAM
# ---------------------------------------------------------

pooled_gradients = tf.reduce_mean(
    gradients,
    axis=(0, 1, 2)
)

conv_outputs = conv_outputs[0]

heatmap = tf.reduce_sum(
    conv_outputs * pooled_gradients,
    axis=-1
)

heatmap = tf.maximum(
    heatmap,
    0
)

max_value = tf.reduce_max(
    heatmap
)

if float(max_value) > 0:
    heatmap = heatmap / max_value

heatmap = heatmap.numpy()


# ---------------------------------------------------------
# Resize heatmap
# ---------------------------------------------------------

heatmap_resized = tf.image.resize(
    heatmap[..., np.newaxis],
    (224, 224)
).numpy().squeeze()


# ---------------------------------------------------------
# Display result
# ---------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)

plt.imshow(original_image)

plt.title(
    f"Original\nActual: {actual_class}"
)

plt.axis("off")


plt.subplot(1, 2, 2)

plt.imshow(original_image)

plt.imshow(
    heatmap_resized,
    alpha=0.45
)

plt.title(
    f"Grad-CAM\n"
    f"Predicted: {predicted_class}\n"
    f"Confidence: {confidence:.2%}"
)

plt.axis("off")

plt.tight_layout()

plt.show()


# ---------------------------------------------------------
# Save result
# ---------------------------------------------------------

GRADCAM_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn",
    "gradcam"
)

os.makedirs(
    GRADCAM_DIR,
    exist_ok=True
)

GRADCAM_PATH = os.path.join(
    GRADCAM_DIR,
    "gradcam_first_test_image.png"
)

plt.savefig(
    GRADCAM_PATH,
    dpi=300,
    bbox_inches="tight"
)

print("\n" + "=" * 70)
print("GRAD-CAM COMPLETED")
print("=" * 70)

print("Actual class    :", actual_class)
print("Predicted class :", predicted_class)
print(f"Confidence      : {confidence:.4f}")
print("Grad-CAM layer  :", last_conv_layer.name)

print("\nSaved to:")
print(GRADCAM_PATH)

print("\nFile exists:", os.path.exists(GRADCAM_PATH))

In [54]:
# WEEK 4 — Step 13: Grad-CAM using Pillow directly

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from PIL import Image

print("=" * 70)
print("GRAD-CAM — SOIL IMAGE EXPLANATION")
print("=" * 70)

# ---------------------------------------------------------
# Test image directory
# ---------------------------------------------------------

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "images",
    "split",
    "test"
)

class_names = [
    "Alluvial_Soil",
    "Arid_Soil",
    "Black_Soil",
    "Laterite_Soil",
    "Mountain_Soil",
    "Red_Soil",
    "Yellow_Soil"
]

# ---------------------------------------------------------
# Find first test image
# ---------------------------------------------------------

image_path = None
actual_class = None

for class_name in class_names:

    class_dir = os.path.join(
        TEST_DIR,
        class_name
    )

    if os.path.exists(class_dir):

        files = sorted([
            f for f in os.listdir(class_dir)
            if f.lower().endswith(
                (".jpg", ".jpeg", ".png")
            )
        ])

        if files:
            image_path = os.path.join(
                class_dir,
                files[0]
            )

            actual_class = class_name
            break

if image_path is None:
    raise FileNotFoundError(
        "No test image was found."
    )

print("Image:", image_path)
print("Actual class:", actual_class)


# ---------------------------------------------------------
# Load image using Pillow
# ---------------------------------------------------------

original_image = Image.open(
    image_path
).convert("RGB")

original_image = original_image.resize(
    (224, 224)
)

img_array = np.array(
    original_image,
    dtype=np.float32
)

input_array = np.expand_dims(
    img_array,
    axis=0
)

# ResNet-50 preprocessing
input_array = tf.keras.applications.resnet50.preprocess_input(
    input_array
)

print("✓ Image loaded and preprocessed")


# ---------------------------------------------------------
# Prediction
# ---------------------------------------------------------

predictions = gradcam_model.predict(
    input_array,
    verbose=0
)

predicted_index = int(
    np.argmax(predictions[0])
)

predicted_class = class_names[
    predicted_index
]

confidence = float(
    predictions[0][predicted_index]
)

print("\nPrediction:")
print("Predicted class:", predicted_class)
print(f"Confidence: {confidence:.4f}")


# ---------------------------------------------------------
# Find last convolutional layer
# ---------------------------------------------------------

last_conv_layer = None

for layer in reversed(resnet_base.layers):

    if isinstance(
        layer,
        tf.keras.layers.Conv2D
    ):
        last_conv_layer = layer
        break

if last_conv_layer is None:
    raise ValueError(
        "Could not find convolutional layer."
    )

print("\nGrad-CAM layer:", last_conv_layer.name)


# ---------------------------------------------------------
# Create Grad-CAM model
# ---------------------------------------------------------

grad_model = tf.keras.Model(
    inputs=gradcam_model.inputs,
    outputs=[
        last_conv_layer.output,
        gradcam_model.output
    ]
)


# ---------------------------------------------------------
# Calculate gradients
# ---------------------------------------------------------

with tf.GradientTape() as tape:

    conv_outputs, predictions_tensor = grad_model(
        input_array
    )

    class_score = predictions_tensor[
        :,
        predicted_index
    ]

gradients = tape.gradient(
    class_score,
    conv_outputs
)


# ---------------------------------------------------------
# Calculate Grad-CAM
# ---------------------------------------------------------

pooled_gradients = tf.reduce_mean(
    gradients,
    axis=(0, 1, 2)
)

conv_outputs = conv_outputs[0]

heatmap = tf.reduce_sum(
    conv_outputs * pooled_gradients,
    axis=-1
)

heatmap = tf.maximum(
    heatmap,
    0
)

max_value = tf.reduce_max(
    heatmap
)

if float(max_value) > 0:
    heatmap = heatmap / max_value

heatmap = heatmap.numpy()


# ---------------------------------------------------------
# Resize heatmap
# ---------------------------------------------------------

heatmap_resized = tf.image.resize(
    heatmap[..., np.newaxis],
    (224, 224)
).numpy().squeeze()


# ---------------------------------------------------------
# Display result
# ---------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)

plt.imshow(original_image)

plt.title(
    f"Original\nActual: {actual_class}"
)

plt.axis("off")


plt.subplot(1, 2, 2)

plt.imshow(original_image)

plt.imshow(
    heatmap_resized,
    alpha=0.45
)

plt.title(
    f"Grad-CAM\n"
    f"Predicted: {predicted_class}\n"
    f"Confidence: {confidence:.2%}"
)

plt.axis("off")

plt.tight_layout()

plt.show()


# ---------------------------------------------------------
# Save result
# ---------------------------------------------------------

GRADCAM_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn",
    "gradcam"
)

os.makedirs(
    GRADCAM_DIR,
    exist_ok=True
)

GRADCAM_PATH = os.path.join(
    GRADCAM_DIR,
    "gradcam_first_test_image.png"
)

plt.savefig(
    GRADCAM_PATH,
    dpi=300,
    bbox_inches="tight"
)

print("\n" + "=" * 70)
print("GRAD-CAM COMPLETED")
print("=" * 70)

print("Actual class    :", actual_class)
print("Predicted class :", predicted_class)
print(f"Confidence      : {confidence:.4f}")
print("Grad-CAM layer  :", last_conv_layer.name)

print("\nSaved to:")
print(GRADCAM_PATH)

print("\nFile exists:", os.path.exists(GRADCAM_PATH))

In [55]:
# WEEK 4 — Step 12: Correct ResNet-50 reconstruction for Grad-CAM

import os
import tensorflow as tf

from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers, Model

print("=" * 70)
print("REBUILDING RESNET-50 MODEL FOR GRAD-CAM")
print("=" * 70)

CNN_MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "models",
    "cnn",
    "resnet50_soil_classifier.keras"
)

print("Saved model:")
print(CNN_MODEL_PATH)
print("Exists:", os.path.exists(CNN_MODEL_PATH))


# ---------------------------------------------------------
# Load the saved classifier
# ---------------------------------------------------------

saved_classifier = tf.keras.models.load_model(
    CNN_MODEL_PATH
)

print("\n✓ Saved classifier loaded")

print("Classifier input :", saved_classifier.input_shape)
print("Classifier output:", saved_classifier.output_shape)


# ---------------------------------------------------------
# Create a NEW ResNet-50 backbone
# ---------------------------------------------------------

resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)

resnet_base.trainable = False

print("\n✓ ResNet-50 backbone loaded")
print("Backbone output:", resnet_base.output_shape)


# ---------------------------------------------------------
# Recreate the classifier architecture
# ---------------------------------------------------------

classifier_head = tf.keras.Sequential(
    [
        layers.Dense(
            128,
            activation="relu",
            name="dense"
        ),

        layers.Dropout(
            0.3,
            name="dropout"
        ),

        layers.Dense(
            7,
            activation="softmax",
            name="dense_1"
        )
    ],
    name="classifier_head"
)


# ---------------------------------------------------------
# Build complete connected model
# ---------------------------------------------------------

inputs = layers.Input(
    shape=(224, 224, 3),
    name="soil_image"
)

# ResNet feature extraction
features = resnet_base(inputs)

# Classifier
outputs = classifier_head(features)

gradcam_model = Model(
    inputs=inputs,
    outputs=outputs,
    name="resnet50_soil_classifier_gradcam"
)


# ---------------------------------------------------------
# Build classifier head before loading weights
# ---------------------------------------------------------

_ = gradcam_model(
    tf.zeros((1, 224, 224, 3))
)


# ---------------------------------------------------------
# Copy trained classifier weights
# ---------------------------------------------------------

classifier_head.set_weights(
    saved_classifier.get_weights()
)

print("\n✓ Trained classifier weights copied")


# ---------------------------------------------------------
# Verify model
# ---------------------------------------------------------

print("\nComplete model:")
print("Input :", gradcam_model.input_shape)
print("Output:", gradcam_model.output_shape)

print("\n" + "=" * 70)
print("GRAD-CAM MODEL READY")
print("=" * 70)

In [56]:
last_conv_layer = resnet_base.get_layer(
    "conv5_block3_out"
)

print("Grad-CAM layer:", last_conv_layer.name)

In [57]:
# WEEK 4 — Step 13: Generate Grad-CAM heatmap

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image

print("=" * 70)
print("GENERATING GRAD-CAM HEATMAP")
print("=" * 70)

# ---------------------------------------------------------
# Test dataset
# ---------------------------------------------------------

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "images",
    "split",
    "test"
)

class_names = [
    "Alluvial_Soil",
    "Arid_Soil",
    "Black_Soil",
    "Laterite_Soil",
    "Mountain_Soil",
    "Red_Soil",
    "Yellow_Soil"
]

# ---------------------------------------------------------
# Find first test image
# ---------------------------------------------------------

image_path = None
actual_class = None

for class_name in class_names:

    class_dir = os.path.join(TEST_DIR, class_name)

    if os.path.exists(class_dir):

        files = sorted([
            f for f in os.listdir(class_dir)
            if f.lower().endswith(
                (".jpg", ".jpeg", ".png")
            )
        ])

        if files:
            image_path = os.path.join(
                class_dir,
                files[0]
            )
            actual_class = class_name
            break

if image_path is None:
    raise FileNotFoundError(
        "No test image found."
    )

print("Image:", image_path)
print("Actual class:", actual_class)


# ---------------------------------------------------------
# Load image with Pillow
# ---------------------------------------------------------

original_image = Image.open(
    image_path
).convert("RGB")

original_image = original_image.resize(
    (224, 224)
)

img_array = np.array(
    original_image,
    dtype=np.float32
)

input_array = np.expand_dims(
    img_array,
    axis=0
)

input_array = tf.keras.applications.resnet50.preprocess_input(
    input_array
)

print("✓ Image loaded")


# ---------------------------------------------------------
# Prediction
# ---------------------------------------------------------

predictions = gradcam_model.predict(
    input_array,
    verbose=0
)

predicted_index = int(
    np.argmax(predictions[0])
)

predicted_class = class_names[predicted_index]

confidence = float(
    predictions[0][predicted_index]
)

print("\nPrediction:")
print("Actual class    :", actual_class)
print("Predicted class :", predicted_class)
print(f"Confidence      : {confidence:.4f}")


# ---------------------------------------------------------
# Correct final convolutional layer
# ---------------------------------------------------------

last_conv_layer = resnet_base.get_layer(
    "conv5_block3_out"
)

print("\nGrad-CAM layer:", last_conv_layer.name)


# ---------------------------------------------------------
# Create Grad-CAM model
# ---------------------------------------------------------

grad_model = tf.keras.Model(
    inputs=gradcam_model.inputs,
    outputs=[
        last_conv_layer.output,
        gradcam_model.output
    ]
)


# ---------------------------------------------------------
# Calculate gradients
# ---------------------------------------------------------

with tf.GradientTape() as tape:

    conv_outputs, predictions_tensor = grad_model(
        input_array
    )

    class_score = predictions_tensor[
        :,
        predicted_index
    ]

gradients = tape.gradient(
    class_score,
    conv_outputs
)


# ---------------------------------------------------------
# Generate heatmap
# ---------------------------------------------------------

pooled_gradients = tf.reduce_mean(
    gradients,
    axis=(0, 1, 2)
)

conv_outputs = conv_outputs[0]

heatmap = tf.reduce_sum(
    conv_outputs * pooled_gradients,
    axis=-1
)

heatmap = tf.maximum(
    heatmap,
    0
)

max_value = tf.reduce_max(
    heatmap
)

if float(max_value) > 0:
    heatmap = heatmap / max_value

heatmap = heatmap.numpy()


# ---------------------------------------------------------
# Resize heatmap
# ---------------------------------------------------------

heatmap_resized = tf.image.resize(
    heatmap[..., np.newaxis],
    (224, 224)
).numpy().squeeze()


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)

plt.imshow(original_image)

plt.title(
    f"Original\nActual: {actual_class}"
)

plt.axis("off")


plt.subplot(1, 2, 2)

plt.imshow(original_image)

plt.imshow(
    heatmap_resized,
    alpha=0.45
)

plt.title(
    f"Grad-CAM\n"
    f"Predicted: {predicted_class}\n"
    f"Confidence: {confidence:.2%}"
)

plt.axis("off")

plt.tight_layout()

plt.show()


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

GRADCAM_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn",
    "gradcam"
)

os.makedirs(
    GRADCAM_DIR,
    exist_ok=True
)

GRADCAM_PATH = os.path.join(
    GRADCAM_DIR,
    "gradcam_first_test_image.png"
)

plt.savefig(
    GRADCAM_PATH,
    dpi=300,
    bbox_inches="tight"
)

print("\n" + "=" * 70)
print("GRAD-CAM COMPLETED")
print("=" * 70)

print("Actual class    :", actual_class)
print("Predicted class :", predicted_class)
print(f"Confidence      : {confidence:.4f}")
print("Grad-CAM layer  :", last_conv_layer.name)
print("\nSaved to:")
print(GRADCAM_PATH)
print("\nFile exists:", os.path.exists(GRADCAM_PATH))

In [58]:
# WEEK 4 — Step 13A: Build a single connected Grad-CAM model

import os
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras import layers

print("=" * 70)
print("BUILDING CONNECTED GRAD-CAM MODEL")
print("=" * 70)

# ---------------------------------------------------------
# Load the saved classifier
# ---------------------------------------------------------

CNN_MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "models",
    "cnn",
    "resnet50_soil_classifier.keras"
)

saved_classifier = tf.keras.models.load_model(
    CNN_MODEL_PATH
)

print("✓ Saved classifier loaded")


# ---------------------------------------------------------
# Create ResNet-50
# ---------------------------------------------------------

resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

resnet_base.trainable = False

print("✓ ResNet-50 loaded")


# ---------------------------------------------------------
# Get the Grad-CAM convolutional layer
# ---------------------------------------------------------

last_conv_layer = resnet_base.get_layer(
    "conv5_block3_out"
)

print("Grad-CAM layer:", last_conv_layer.name)


# ---------------------------------------------------------
# Create ONE connected computation graph
# ---------------------------------------------------------

inputs = tf.keras.Input(
    shape=(224, 224, 3),
    name="soil_image"
)

# Run image through ResNet
resnet_output = resnet_base(inputs)

# Get convolutional features directly from the same graph
conv_output = last_conv_layer.output

# IMPORTANT:
# Use the ResNet pooled output for the classifier
pooled_features = layers.GlobalAveragePooling2D(
    name="gradcam_global_average_pooling"
)(resnet_base.layers[-1].output)

# Create classifier head
dense_output = layers.Dense(
    128,
    activation="relu",
    name="dense"
)(pooled_features)

dropout_output = layers.Dropout(
    0.3,
    name="dropout"
)(dense_output)

final_output = layers.Dense(
    7,
    activation="softmax",
    name="dense_1"
)(dropout_output)


# ---------------------------------------------------------
# Build the model
# ---------------------------------------------------------

gradcam_model = tf.keras.Model(
    inputs=inputs,
    outputs=[
        conv_output,
        final_output
    ],
    name="soil_resnet50_gradcam"
)


# ---------------------------------------------------------
# Build once
# ---------------------------------------------------------

dummy_input = tf.zeros(
    (1, 224, 224, 3)
)

_ = gradcam_model(
    dummy_input
)


# ---------------------------------------------------------
# Copy trained classifier weights
# ---------------------------------------------------------

trained_weights = saved_classifier.get_weights()

print("\nSaved classifier weight arrays:",
      len(trained_weights))

print("New classifier weight arrays:",
      len([
          w for w in gradcam_model.layers
          if w.get_weights()
      ]))


# The classifier consists of:
# Dense(128), Dropout, Dense(7)
#
# Locate them in the new model.

new_dense = gradcam_model.get_layer("dense")
new_output = gradcam_model.get_layer("dense_1")

# saved_classifier layers contain the trained Dense layers
saved_dense_layers = [
    layer for layer in saved_classifier.layers
    if isinstance(
        layer,
        tf.keras.layers.Dense
    )
]

print("\nSaved Dense layers:",
      [layer.name for layer in saved_dense_layers])

# Copy Dense weights
new_dense.set_weights(
    saved_dense_layers[0].get_weights()
)

new_output.set_weights(
    saved_dense_layers[1].get_weights()
)

print("✓ Trained classifier weights copied")


# ---------------------------------------------------------
# Verify
# ---------------------------------------------------------

test_result = gradcam_model.predict(
    dummy_input,
    verbose=0
)

print("\nConvolution output shape:",
      test_result[0].shape)

print("Prediction output shape:",
      test_result[1].shape)

print("\n" + "=" * 70)
print("CONNECTED GRAD-CAM MODEL READY")
print("=" * 70)

In [59]:
print("=" * 70)
print("SAVED CNN MODEL STRUCTURE")
print("=" * 70)

for i, layer in enumerate(saved_classifier.layers):
    print(
        i,
        "|",
        layer.name,
        "|",
        type(layer).__name__,
        "|",
        layer.input_shape if hasattr(layer, "input_shape") else "",
        "|",
        layer.output_shape if hasattr(layer, "output_shape") else ""
    )

In [60]:
print("=" * 70)
print("SAVED CNN MODEL STRUCTURE")
print("=" * 70)

for i, layer in enumerate(saved_classifier.layers):
    print(i, "|", layer.name, "|", type(layer).__name__)

In [61]:
print("=" * 70)
print("SAVED CLASSIFIER WEIGHTS")
print("=" * 70)

for layer in saved_classifier.layers:
    weights = layer.get_weights()

    if weights:
        print(
            layer.name,
            "→",
            [w.shape for w in weights]
        )

In [62]:
# WEEK 4 — Step 13: Final Grad-CAM model

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.models import Model

print("=" * 70)
print("BUILDING FINAL GRAD-CAM MODEL")
print("=" * 70)

# ---------------------------------------------------------
# 1. Load saved classifier
# ---------------------------------------------------------

CNN_MODEL_PATH = os.path.join(
    PROJECT_DIR,
    "models",
    "cnn",
    "resnet50_soil_classifier.keras"
)

saved_classifier = tf.keras.models.load_model(
    CNN_MODEL_PATH
)

print("✓ Saved classifier loaded")


# ---------------------------------------------------------
# 2. Get trained Dense weights
# ---------------------------------------------------------

dense_1_weights = saved_classifier.get_layer(
    "dense_1"
).get_weights()

dense_2_weights = saved_classifier.get_layer(
    "dense_2"
).get_weights()

print("✓ Classifier weights extracted")

print(
    "Dense 1 weights:",
    [w.shape for w in dense_1_weights]
)

print(
    "Dense 2 weights:",
    [w.shape for w in dense_2_weights]
)


# ---------------------------------------------------------
# 3. ResNet-50 WITHOUT pooling
# ---------------------------------------------------------

resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

resnet_base.trainable = False

print("✓ ResNet-50 loaded")


# ---------------------------------------------------------
# 4. Build connected Grad-CAM model
# ---------------------------------------------------------

inputs = Input(
    shape=(224, 224, 3),
    name="soil_image"
)

# Convolutional feature maps
conv_features = resnet_base(
    inputs
)

# This is the exact layer output needed for Grad-CAM
last_conv_output = resnet_base.get_layer(
    "conv5_block3_out"
).output

# Global average pooling
pooled = GlobalAveragePooling2D(
    name="global_average_pooling"
)(conv_features)

# Same classifier architecture used in Week 3
x = Dense(
    128,
    activation="relu",
    name="dense_1"
)(pooled)

x = Dropout(
    0.3,
    name="dropout_1"
)(x)

outputs = Dense(
    7,
    activation="softmax",
    name="dense_2"
)(x)


# ---------------------------------------------------------
# 5. Create model returning both outputs
# ---------------------------------------------------------

gradcam_model = Model(
    inputs=inputs,
    outputs=[
        last_conv_output,
        outputs
    ],
    name="resnet50_gradcam"
)


# ---------------------------------------------------------
# 6. Build model
# ---------------------------------------------------------

dummy_input = tf.zeros(
    (1, 224, 224, 3)
)

_ = gradcam_model(
    dummy_input
)


# ---------------------------------------------------------
# 7. Copy trained weights
# ---------------------------------------------------------

gradcam_model.get_layer(
    "dense_1"
).set_weights(
    dense_1_weights
)

gradcam_model.get_layer(
    "dense_2"
).set_weights(
    dense_2_weights
)

print("✓ Trained classifier weights restored")


# ---------------------------------------------------------
# 8. Verify outputs
# ---------------------------------------------------------

conv_test, prediction_test = gradcam_model.predict(
    dummy_input,
    verbose=0
)

print("\nConvolution output shape:",
      conv_test.shape)

print("Prediction output shape:",
      prediction_test.shape)

print("\n" + "=" * 70)
print("FINAL GRAD-CAM MODEL READY")
print("=" * 70)

In [63]:
# STEP 13A — Recreate the ResNet-50 feature extractor only

import tensorflow as tf
from tensorflow.keras.applications import ResNet50

print("=" * 70)
print("RECREATING RESNET-50 FEATURE EXTRACTOR")
print("=" * 70)

resnet_base = ResNet50(
    weights="imagenet",
    include_top=False,
    pooling=None,
    input_shape=(224, 224, 3)
)

resnet_base.trainable = False

print("✓ ResNet-50 loaded")

print("Input shape :", resnet_base.input_shape)
print("Output shape:", resnet_base.output_shape)

# Verify a real tensor
dummy_image = tf.zeros((1, 224, 224, 3))

feature_maps = resnet_base(dummy_image)

print("Feature maps shape:", feature_maps.shape)

# Check Grad-CAM layer
last_conv_layer = resnet_base.get_layer("conv5_block3_out")

print("Grad-CAM layer:", last_conv_layer.name)
print("Grad-CAM output shape:", last_conv_layer.output.shape)

print("\n" + "=" * 70)
print("RESNET-50 BACKBONE VERIFIED")
print("=" * 70)

In [64]:
# WEEK 4 — Step 13B: Grad-CAM heatmap

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image

print("=" * 70)
print("GENERATING GRAD-CAM")
print("=" * 70)

# ---------------------------------------------------------
# Paths and class names
# ---------------------------------------------------------

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "images",
    "split",
    "test"
)

class_names = [
    "Alluvial_Soil",
    "Arid_Soil",
    "Black_Soil",
    "Laterite_Soil",
    "Mountain_Soil",
    "Red_Soil",
    "Yellow_Soil"
]

# ---------------------------------------------------------
# Find first test image
# ---------------------------------------------------------

image_path = None
actual_class = None

for class_name in class_names:

    class_dir = os.path.join(TEST_DIR, class_name)

    if os.path.exists(class_dir):

        files = sorted([
            f for f in os.listdir(class_dir)
            if f.lower().endswith(
                (".jpg", ".jpeg", ".png")
            )
        ])

        if files:
            image_path = os.path.join(
                class_dir,
                files[0]
            )
            actual_class = class_name
            break

if image_path is None:
    raise FileNotFoundError("No test image found.")

print("Image:", image_path)
print("Actual class:", actual_class)


# ---------------------------------------------------------
# Load image
# ---------------------------------------------------------

original_image = Image.open(
    image_path
).convert("RGB")

original_image = original_image.resize(
    (224, 224)
)

img_array = np.asarray(
    original_image,
    dtype=np.float32
)

input_array = np.expand_dims(
    img_array,
    axis=0
)

input_array = tf.keras.applications.resnet50.preprocess_input(
    input_array
)

print("✓ Image loaded")


# ---------------------------------------------------------
# Load classifier
# ---------------------------------------------------------

classifier = tf.keras.models.load_model(
    os.path.join(
        PROJECT_DIR,
        "models",
        "cnn",
        "resnet50_soil_classifier.keras"
    )
)

print("✓ Classifier loaded")


# ---------------------------------------------------------
# Get classifier weights
# ---------------------------------------------------------

dense1 = classifier.get_layer("dense_1")
dense2 = classifier.get_layer("dense_2")

W1, b1 = dense1.get_weights()
W2, b2 = dense2.get_weights()

print("✓ Classifier weights loaded")


# ---------------------------------------------------------
# Grad-CAM calculation
# ---------------------------------------------------------

with tf.GradientTape() as tape:

    # Get convolutional feature maps
    conv_maps = resnet_base(input_array, training=False)

    # Watch feature maps
    tape.watch(conv_maps)

    # Global average pooling
    pooled = tf.reduce_mean(
        conv_maps,
        axis=[1, 2]
    )

    # Dense 1
    dense_output = tf.nn.relu(
        tf.matmul(
            pooled,
            W1
        ) + b1
    )

    # Dense 2
    predictions = tf.nn.softmax(
        tf.matmul(
            dense_output,
            W2
        ) + b2
    )

    predicted_index = tf.argmax(
        predictions[0]
    )

    class_score = predictions[
        0,
        predicted_index
    ]


# ---------------------------------------------------------
# Gradients
# ---------------------------------------------------------

gradients = tape.gradient(
    class_score,
    conv_maps
)

# Global-average gradient weights
weights = tf.reduce_mean(
    gradients,
    axis=[1, 2]
)

# Weighted feature maps
cam = tf.reduce_sum(
    conv_maps * weights[:, None, None, :],
    axis=-1
)

# ReLU
cam = tf.maximum(
    cam,
    0
)

# Normalize
cam_max = tf.reduce_max(cam)

if float(cam_max) > 0:
    cam = cam / cam_max

heatmap = cam[0].numpy()


# ---------------------------------------------------------
# Prediction information
# ---------------------------------------------------------

predicted_index = int(predicted_index.numpy())

predicted_class = class_names[
    predicted_index
]

confidence = float(
    predictions[0, predicted_index].numpy()
)

print("\nPrediction:")
print("Actual class    :", actual_class)
print("Predicted class :", predicted_class)
print(f"Confidence      : {confidence:.4f}")


# ---------------------------------------------------------
# Resize heatmap
# ---------------------------------------------------------

heatmap_resized = tf.image.resize(
    heatmap[..., np.newaxis],
    (224, 224)
).numpy().squeeze()


# ---------------------------------------------------------
# Display
# ---------------------------------------------------------

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)

plt.imshow(original_image)

plt.title(
    f"Original\nActual: {actual_class}"
)

plt.axis("off")


plt.subplot(1, 2, 2)

plt.imshow(original_image)

plt.imshow(
    heatmap_resized,
    alpha=0.45
)

plt.title(
    f"Grad-CAM\n"
    f"Predicted: {predicted_class}\n"
    f"Confidence: {confidence:.2%}"
)

plt.axis("off")

plt.tight_layout()

plt.show()


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

GRADCAM_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "cnn",
    "gradcam"
)

os.makedirs(
    GRADCAM_DIR,
    exist_ok=True
)

GRADCAM_PATH = os.path.join(
    GRADCAM_DIR,
    "gradcam_first_test_image.png"
)

plt.savefig(
    GRADCAM_PATH,
    dpi=300,
    bbox_inches="tight"
)

print("\n" + "=" * 70)
print("GRAD-CAM COMPLETED")
print("=" * 70)

print("Saved to:")
print(GRADCAM_PATH)

print("File exists:", os.path.exists(GRADCAM_PATH))

In [65]:
# WEEK 4 — Step 14B: Hybrid Analysis Function

import os
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from xgboost import XGBRegressor

print("=" * 70)
print("HYBRID ANALYSIS")
print("=" * 70)

# ---------------------------------------------------------
# 1. Load ResNet-50 backbone
# ---------------------------------------------------------

resnet_base = tf.keras.applications.ResNet50(
    weights="imagenet",
    include_top=False,
    pooling="avg",
    input_shape=(224, 224, 3)
)

resnet_base.trainable = False

print("✓ ResNet-50 loaded")


# ---------------------------------------------------------
# 2. Load trained CNN classifier
# ---------------------------------------------------------

cnn_model_path = os.path.join(
    PROJECT_DIR,
    "models",
    "cnn",
    "resnet50_soil_classifier.keras"
)

cnn_classifier = tf.keras.models.load_model(
    cnn_model_path
)

print("✓ CNN classifier loaded")


# ---------------------------------------------------------
# 3. Load XGBoost models
# ---------------------------------------------------------

xgb_n = XGBRegressor()
xgb_p = XGBRegressor()
xgb_k = XGBRegressor()

xgb_n.load_model(
    os.path.join(
        PROJECT_DIR,
        "models",
        "xgboost",
        "xgboost_nitrogen_final.json"
    )
)

xgb_p.load_model(
    os.path.join(
        PROJECT_DIR,
        "models",
        "xgboost",
        "xgboost_phosphorus_final.json"
    )
)

xgb_k.load_model(
    os.path.join(
        PROJECT_DIR,
        "models",
        "xgboost",
        "xgboost_potassium_final.json"
    )
)

print("✓ XGBoost N model loaded")
print("✓ XGBoost P model loaded")
print("✓ XGBoost K model loaded")


# ---------------------------------------------------------
# 4. Class names
# ---------------------------------------------------------

class_names = [
    "Alluvial_Soil",
    "Arid_Soil",
    "Black_Soil",
    "Laterite_Soil",
    "Mountain_Soil",
    "Red_Soil",
    "Yellow_Soil"
]


# ---------------------------------------------------------
# 5. Hybrid analysis function
# ---------------------------------------------------------

def hybrid_soil_analysis(
    image_path,
    moisture,
    ph,
    organic_matter,
    nitrogen,
    phosphorus,
    potassium
):

    # =====================================================
    # IMAGE BRANCH — ResNet-50
    # =====================================================

    image = Image.open(
        image_path
    ).convert("RGB")

    image = image.resize(
        (224, 224)
    )

    image_array = np.asarray(
        image,
        dtype=np.float32
    )

    image_array = np.expand_dims(
        image_array,
        axis=0
    )

    image_array = (
        tf.keras.applications.resnet50.preprocess_input(
            image_array
        )
    )

    # Extract ResNet-50 features
    features = resnet_base(
        image_array,
        training=False
    )

    # CNN prediction
    probabilities = cnn_classifier.predict(
        features,
        verbose=0
    )[0]

    predicted_index = int(
        np.argmax(probabilities)
    )

    predicted_class = class_names[
        predicted_index
    ]

    cnn_confidence = float(
        probabilities[predicted_index]
    )


    # =====================================================
    # STRUCTURED DATA BRANCH — XGBoost
    # =====================================================

    # Nitrogen prediction
    X_n = pd.DataFrame(
        [[
            moisture,
            ph,
            organic_matter,
            phosphorus,
            potassium
        ]],
        columns=[
            "moisture",
            "ph",
            "organic_matter",
            "phosphorus",
            "potassium"
        ]
    )

    predicted_n = float(
        xgb_n.predict(X_n)[0]
    )


    # Phosphorus prediction
    X_p = pd.DataFrame(
        [[
            moisture,
            ph,
            organic_matter,
            nitrogen,
            potassium
        ]],
        columns=[
            "moisture",
            "ph",
            "organic_matter",
            "nitrogen",
            "potassium"
        ]
    )

    predicted_p = float(
        xgb_p.predict(X_p)[0]
    )


    # Potassium prediction
    X_k = pd.DataFrame(
        [[
            moisture,
            ph,
            organic_matter,
            nitrogen,
            phosphorus
        ]],
        columns=[
            "moisture",
            "ph",
            "organic_matter",
            "nitrogen",
            "phosphorus"
        ]
    )

    predicted_k = float(
        xgb_k.predict(X_k)[0]
    )


    # =====================================================
    # HYBRID RESULT
    # =====================================================

    result = {

        "image_analysis": {
            "predicted_soil_class": predicted_class,
            "cnn_confidence": cnn_confidence
        },

        "structured_analysis": {

            "input_nitrogen": float(nitrogen),
            "predicted_nitrogen": predicted_n,

            "input_phosphorus": float(phosphorus),
            "predicted_phosphorus": predicted_p,

            "input_potassium": float(potassium),
            "predicted_potassium": predicted_k,

            "moisture": float(moisture),
            "ph": float(ph),
            "organic_matter": float(organic_matter)
        },

        "hybrid_analysis": {

            "visual_soil_class": predicted_class,

            "image_confidence": cnn_confidence,

            "nutrient_analysis_available": True,

            "health_score": None,

            "health_score_status":
                "Not calculated because "
                "validated nutrient-health thresholds "
                "are not provided in the project dataset."
        }
    }

    return result


print("✓ Hybrid analysis function created")

In [66]:
# WEEK 4 — Step 14C: Test Hybrid Pipeline

# ---------------------------------------------------------
# Select one real soil-test record
# ---------------------------------------------------------

sample = soil_df.iloc[0]

print("=" * 70)
print("SELECTED SOIL-TEST RECORD")
print("=" * 70)

print("Sample ID       :", sample["sample_id"])
print("Crop            :", sample["crop"])
print("Moisture        :", sample["moisture"])
print("pH              :", sample["ph"])
print("Organic Matter  :", sample["organic_matter"])
print("Nitrogen        :", sample["nitrogen"])
print("Phosphorus      :", sample["phosphorus"])
print("Potassium       :", sample["potassium"])


# ---------------------------------------------------------
# Find one real test image
# ---------------------------------------------------------

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "data",
    "images",
    "split",
    "test"
)

image_path = None
actual_class = None

for class_name in class_names:

    class_dir = os.path.join(
        TEST_DIR,
        class_name
    )

    if os.path.exists(class_dir):

        image_files = sorted([
            f for f in os.listdir(class_dir)
            if f.lower().endswith(
                (".jpg", ".jpeg", ".png")
            )
        ])

        if image_files:
            image_path = os.path.join(
                class_dir,
                image_files[0]
            )
            actual_class = class_name
            break


print("\nImage used:")
print(image_path)
print("Actual image class:", actual_class)


# ---------------------------------------------------------
# Run hybrid analysis
# ---------------------------------------------------------

hybrid_result = hybrid_soil_analysis(

    image_path=image_path,

    moisture=sample["moisture"],
    ph=sample["ph"],
    organic_matter=sample["organic_matter"],
    nitrogen=sample["nitrogen"],
    phosphorus=sample["phosphorus"],
    potassium=sample["potassium"]
)


# ---------------------------------------------------------
# Display result
# ---------------------------------------------------------

print("\n" + "=" * 70)
print("HYBRID SOIL ASSESSMENT")
print("=" * 70)

print("\nIMAGE ANALYSIS")
print("-" * 40)

print(
    "Predicted Soil Class :",
    hybrid_result["image_analysis"]
    ["predicted_soil_class"]
)

print(
    "CNN Confidence       :",
    f"{hybrid_result['image_analysis']['cnn_confidence']:.2%}"
)


print("\nSTRUCTURED ANALYSIS")
print("-" * 40)

structured = hybrid_result[
    "structured_analysis"
]

print(
    "Input Nitrogen       :",
    structured["input_nitrogen"]
)

print(
    "Predicted Nitrogen   :",
    structured["predicted_nitrogen"]
)

print(
    "Input Phosphorus     :",
    structured["input_phosphorus"]
)

print(
    "Predicted Phosphorus :",
    structured["predicted_phosphorus"]
)

print(
    "Input Potassium      :",
    structured["input_potassium"]
)

print(
    "Predicted Potassium  :",
    structured["predicted_potassium"]
)


print("\nHYBRID ANALYSIS")
print("-" * 40)

print(
    "Visual Soil Class    :",
    hybrid_result["hybrid_analysis"]
    ["visual_soil_class"]
)

print(
    "Image Confidence     :",
    f"{hybrid_result['hybrid_analysis']['image_confidence']:.2%}"
)

print(
    "Nutrient Analysis    :",
    hybrid_result["hybrid_analysis"]
    ["nutrient_analysis_available"]
)

print(
    "Health Score         :",
    hybrid_result["hybrid_analysis"]
    ["health_score"]
)

print("\n" + "=" * 70)
print("HYBRID PIPELINE TEST COMPLETED")
print("=" * 70)

In [67]:
# WEEK 4 — Step 15: Save Hybrid Analysis Report

import os
from datetime import datetime

REPORT_DIR = os.path.join(
    PROJECT_DIR,
    "reports",
    "hybrid"
)

os.makedirs(
    REPORT_DIR,
    exist_ok=True
)

REPORT_PATH = os.path.join(
    REPORT_DIR,
    "Week_4_Hybrid_Analysis_Report.md"
)


# ---------------------------------------------------------
# Extract values
# ---------------------------------------------------------

image_result = hybrid_result["image_analysis"]
structured_result = hybrid_result["structured_analysis"]
hybrid_info = hybrid_result["hybrid_analysis"]


# ---------------------------------------------------------
# Create report
# ---------------------------------------------------------

report = f"""# Week 4 — Hybrid Soil Analysis Report

## 1. Objective

This module combines:

- ResNet-50 image-based soil classification
- XGBoost structured soil-test analysis
- Explainable AI outputs from Grad-CAM and SHAP

The purpose is to provide a combined soil assessment using both
visual soil information and structured soil-test parameters.

---

## 2. Image-Based Analysis

**Actual Image Class:** {actual_class}

**Predicted Soil Class:** {image_result["predicted_soil_class"]}

**CNN Confidence:** {image_result["cnn_confidence"]:.4f}

The ResNet-50 model provides the visual soil-class prediction.

---

## 3. Structured Soil-Test Analysis

### Input Parameters

| Parameter | Value |
|---|---:|
| Moisture | {structured_result["moisture"]:.4f} |
| pH | {structured_result["ph"]:.4f} |
| Organic Matter | {structured_result["organic_matter"]:.4f} |
| Nitrogen | {structured_result["input_nitrogen"]:.4f} |
| Phosphorus | {structured_result["input_phosphorus"]:.4f} |
| Potassium | {structured_result["input_potassium"]:.4f} |

### XGBoost Predictions

| Nutrient | Input Value | XGBoost Prediction |
|---|---:|---:|
| Nitrogen | {structured_result["input_nitrogen"]:.4f} | {structured_result["predicted_nitrogen"]:.4f} |
| Phosphorus | {structured_result["input_phosphorus"]:.4f} | {structured_result["predicted_phosphorus"]:.4f} |
| Potassium | {structured_result["input_potassium"]:.4f} | {structured_result["predicted_potassium"]:.4f} |

---

## 4. Hybrid Assessment

The hybrid system combines the outputs at the assessment level:

**Image branch:**

ResNet-50 → Soil Class + CNN Confidence

**Structured branch:**

XGBoost → Nitrogen + Phosphorus + Potassium predictions

**Combined output:**

Visual Soil Classification + Quantitative Nutrient Analysis

This approach avoids treating soil-class predictions and nutrient
regression values as the same type of output.

---

## 5. Confidence Handling

The CNN confidence is reported directly with the predicted soil class.

The XGBoost models provide continuous nutrient predictions.
Their reliability is represented separately through the model
evaluation metrics rather than by treating regression output as
classification probability.

---

## 6. Soil Health Score

**Status: Pending validated scoring thresholds.**

The available project dataset does not provide validated nutrient
deficiency classes or a soil-health scoring formula. Therefore, an
arbitrary threshold-based health score has not been introduced.

A final soil-health score should be implemented only after validated
thresholds/rules are supplied by the project/mentor or an appropriate
agronomic reference is formally selected.

---

## 7. Explainable AI

### CNN

Grad-CAM was implemented to visualize image regions contributing
to the CNN prediction.

### Structured ML

SHAP was implemented to explain the contribution of structured
soil-test features to the XGBoost predictions.

---

## 8. Models Used

- ResNet-50 — image feature extraction and soil classification
- Dense neural-network classifier — classification head
- XGBoost — structured N/P/K regression
- Grad-CAM — CNN explanation
- SHAP — XGBoost explanation

---

## 9. Sample Information

**Sample ID:** {sample["sample_id"]}

**Crop:** {sample["crop"]}

**Image:** {image_path}

**Report Generated:** {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

---

## 10. Conclusion

The ResNet-50 and XGBoost branches have been successfully integrated
into a hybrid soil-analysis pipeline.

The system can currently provide:

1. Visual soil classification
2. CNN confidence
3. Quantitative N/P/K predictions
4. Grad-CAM visual explanation
5. SHAP feature explanations

Soil-health scoring remains pending validated agronomic thresholds.
"""


# ---------------------------------------------------------
# Save report
# ---------------------------------------------------------

with open(
    REPORT_PATH,
    "w",
    encoding="utf-8"
) as f:
    f.write(report)


print("=" * 70)
print("HYBRID REPORT SAVED")
print("=" * 70)

print("Report path:")
print(REPORT_PATH)

print("\nFile exists:", os.path.exists(REPORT_PATH))
print("Report size:", os.path.getsize(REPORT_PATH), "bytes")

In [68]:
# WEEK 4 — Step 16: Final Validation Checklist

import os

print("=" * 75)
print("WEEK 4 — FINAL VALIDATION CHECKLIST")
print("=" * 75)

checks = {
    # Structured ML
    "Clean soil-test dataset":
        os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "data",
                "soil_tests",
                "processed",
                "clean_soil_test_data.csv"
            )
        ),

    "XGBoost Nitrogen model":
        os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "models",
                "xgboost",
                "xgboost_nitrogen_final.json"
            )
        ),

    "XGBoost Phosphorus model":
        os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "models",
                "xgboost",
                "xgboost_phosphorus_final.json"
            )
        ),

    "XGBoost Potassium model":
        os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "models",
                "xgboost",
                "xgboost_potassium_final.json"
            )
        ),

    # XAI
    "SHAP Nitrogen plot":
        os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "reports",
                "xgboost",
                "shap",
                "shap_nitrogen_summary.png"
            )
        ),

    "SHAP Phosphorus plot":
        os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "reports",
                "xgboost",
                "shap",
                "shap_phosphorus_summary.png"
            )
        ),

    "SHAP Potassium plot":
        os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "reports",
                "xgboost",
                "shap",
                "shap_potassium_summary.png"
            )
        ),

    # CNN XAI
    "Grad-CAM plot":
        os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "reports",
                "cnn",
                "gradcam",
                "gradcam_first_test_image.png"
            )
        ),

    # Hybrid
    "Hybrid analysis report":
        os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "reports",
                "hybrid",
                "Week_4_Hybrid_Analysis_Report.md"
            )
        ),

    # CNN model
    "ResNet-50 classifier":
        os.path.exists(
            os.path.join(
                PROJECT_DIR,
                "models",
                "cnn",
                "resnet50_soil_classifier.keras"
            )
        )
}


# ---------------------------------------------------------
# Display results
# ---------------------------------------------------------

for item, status in checks.items():

    symbol = "✓" if status else "✗"

    print(
        f"{symbol} {item}"
    )


# ---------------------------------------------------------
# Overall status
# ---------------------------------------------------------

passed = sum(checks.values())
total = len(checks)

print("\n" + "=" * 75)
print(f"RESULT: {passed}/{total} checks passed")
print("=" * 75)

if passed == total:
    print("✓ All required project artifacts are present.")
else:
    print("⚠ Some artifacts are missing.")
    print("We will fix missing artifacts before moving forward.")

In [69]:
# WEEK 4 — Step 17A
# Final Week 4 Summary Report

import os
from datetime import datetime

summary_dir = os.path.join(
    PROJECT_DIR,
    "reports",
    "week4"
)

os.makedirs(summary_dir, exist_ok=True)

summary_path = os.path.join(
    summary_dir,
    "Week_4_Final_Summary.md"
)

summary = """# Week 4 — Final Summary

## Project
AI-Powered Soil Analytics System for Nutrient Assessment and Intelligent Crop Advisory

---

## 1. Structured ML

The cleaned soil-test dataset contains 146 records and six
required numeric soil parameters:

- Moisture
- pH
- Organic Matter
- Nitrogen
- Phosphorus
- Potassium

XGBoost regression models were developed for:

- Nitrogen
- Phosphorus
- Potassium

The models were evaluated using MAE, RMSE and R².

Hyperparameter tuning was performed using GridSearchCV with
5-fold cross-validation.

Final models were saved under:

models/xgboost/

---

## 2. Final XGBoost Test Performance

### Nitrogen

MAE: 0.0571
RMSE: 0.1256
R²: 0.4978

### Phosphorus

MAE: 35.8509
RMSE: 68.4390
R²: 0.4947

### Potassium

MAE: 227.4419
RMSE: 334.8574
R²: 0.0062

The results indicate that the models have different predictive
strengths, with nitrogen and phosphorus showing more useful
predictive performance than potassium on the held-out test set.

---

## 3. Explainable AI

### CNN — Grad-CAM

Grad-CAM was implemented for the ResNet-50 image classification
branch.

A high-confidence incorrect prediction was also inspected to
demonstrate model attention during an error case.

### XGBoost — SHAP

SHAP explanations were generated for the:

- Nitrogen model
- Phosphorus model
- Potassium model

These explanations show the contribution of structured soil-test
features to the model predictions.

---

## 4. Hybrid Analysis

The hybrid system combines:

ResNet-50 image analysis

+

XGBoost structured soil-test analysis

The image branch provides:

- Soil class
- CNN confidence

The structured branch provides:

- Nitrogen prediction
- Phosphorus prediction
- Potassium prediction

The outputs are combined at the soil-assessment level rather than
mathematically averaging incompatible output types.

---

## 5. Soil Health Score

A final numerical soil-health score has not been assigned.

Reason:

The available project dataset and provided milestone material do
not specify validated nutrient-deficiency thresholds or a numerical
soil-health scoring formula.

Therefore, no arbitrary threshold or score has been introduced.

A validated scoring method can be added when the required
agronomic thresholds/rules are provided.

---

## 6. Models and Methods

### Image Analysis

ResNet-50 transfer learning

### Structured Analysis

XGBoost regression

### CNN Explainability

Grad-CAM

### Structured Model Explainability

SHAP

### Hybrid Analysis

ResNet-50 + XGBoost

---

## 7. Week 4 Deliverables

- Trained CNN model
- CNN evaluation results
- Confusion matrix
- Grad-CAM visualization
- XGBoost N model
- XGBoost P model
- XGBoost K model
- XGBoost evaluation results
- SHAP explanations
- Hybrid analysis function
- Hybrid analysis report

---

## 8. Conclusion

Week 4 machine-learning, explainability and hybrid-analysis
implementation has been completed based on the available project
data and milestone requirements.

The remaining soil-health scoring component requires validated
agronomic thresholds before a final numerical score can be
scientifically implemented.
"""

with open(
    summary_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(summary)

print("=" * 70)
print("WEEK 4 SUMMARY CREATED")
print("=" * 70)

print("Saved:")
print(summary_path)

print("Exists:", os.path.exists(summary_path))

In [70]:
# HYBRID TEST — 10 REAL SOIL-TEST RECORDS

import os
import pandas as pd

results = []

# Use first 10 real records
for i in range(min(10, len(soil_df))):

    sample = soil_df.iloc[i]

    result = hybrid_soil_analysis(
        image_path=image_path,
        moisture=sample["moisture"],
        ph=sample["ph"],
        organic_matter=sample["organic_matter"],
        nitrogen=sample["nitrogen"],
        phosphorus=sample["phosphorus"],
        potassium=sample["potassium"]
    )

    results.append({
        "sample_id": sample["sample_id"],
        "crop": sample["crop"],

        "soil_class":
            result["image_analysis"]["predicted_soil_class"],

        "cnn_confidence":
            result["image_analysis"]["cnn_confidence"],

        "input_nitrogen":
            sample["nitrogen"],

        "predicted_nitrogen":
            result["structured_analysis"]["predicted_nitrogen"],

        "input_phosphorus":
            sample["phosphorus"],

        "predicted_phosphorus":
            result["structured_analysis"]["predicted_phosphorus"],

        "input_potassium":
            sample["potassium"],

        "predicted_potassium":
            result["structured_analysis"]["predicted_potassium"]
    })


# Convert to table
hybrid_test_df = pd.DataFrame(results)

print("=" * 80)
print("HYBRID SYSTEM — 10 REAL SAMPLE TEST")
print("=" * 80)

display(hybrid_test_df)


# ---------------------------------------------------------
# Save results
# ---------------------------------------------------------

hybrid_dir = os.path.join(
    PROJECT_DIR,
    "reports",
    "hybrid"
)

os.makedirs(hybrid_dir, exist_ok=True)

hybrid_csv_path = os.path.join(
    hybrid_dir,
    "hybrid_test_results.csv"
)

hybrid_test_df.to_csv(
    hybrid_csv_path,
    index=False
)

print("\n✓ Results saved")
print(hybrid_csv_path)

print("\nRows tested:", len(hybrid_test_df))
print("File exists:", os.path.exists(hybrid_csv_path))

In [71]:
# HYBRID TEST — DIFFERENT REAL IMAGES + REAL SOIL-TEST RECORDS

import os
import pandas as pd

results = []

# ---------------------------------------------------------
# Collect real test images
# ---------------------------------------------------------

test_images = []

for class_name in class_names:

    class_dir = os.path.join(
        TEST_DIR,
        class_name
    )

    if os.path.exists(class_dir):

        files = sorted([
            f for f in os.listdir(class_dir)
            if f.lower().endswith(
                (".jpg", ".jpeg", ".png")
            )
        ])

        for file in files:
            test_images.append({
                "path": os.path.join(class_dir, file),
                "actual_class": class_name
            })


# ---------------------------------------------------------
# Use 10 different real images
# ---------------------------------------------------------

num_tests = min(
    10,
    len(soil_df),
    len(test_images)
)

for i in range(num_tests):

    sample = soil_df.iloc[i]
    image_info = test_images[i]

    result = hybrid_soil_analysis(

        image_path=image_info["path"],

        moisture=sample["moisture"],
        ph=sample["ph"],
        organic_matter=sample["organic_matter"],
        nitrogen=sample["nitrogen"],
        phosphorus=sample["phosphorus"],
        potassium=sample["potassium"]
    )

    results.append({

        "sample_id": sample["sample_id"],

        "crop": sample["crop"],

        "image": os.path.basename(
            image_info["path"]
        ),

        "actual_image_class":
            image_info["actual_class"],

        "predicted_soil_class":
            result["image_analysis"]
            ["predicted_soil_class"],

        "cnn_confidence":
            result["image_analysis"]
            ["cnn_confidence"],

        "input_nitrogen":
            sample["nitrogen"],

        "predicted_nitrogen":
            result["structured_analysis"]
            ["predicted_nitrogen"],

        "input_phosphorus":
            sample["phosphorus"],

        "predicted_phosphorus":
            result["structured_analysis"]
            ["predicted_phosphorus"],

        "input_potassium":
            sample["potassium"],

        "predicted_potassium":
            result["structured_analysis"]
            ["predicted_potassium"]
    })


# ---------------------------------------------------------
# Create results table
# ---------------------------------------------------------

hybrid_real_test_df = pd.DataFrame(results)

print("=" * 90)
print("HYBRID VALIDATION — DIFFERENT REAL IMAGES")
print("=" * 90)

display(hybrid_real_test_df)


# ---------------------------------------------------------
# Calculate image accuracy for these 10 cases
# ---------------------------------------------------------

image_correct = (
    hybrid_real_test_df[
        "actual_image_class"
    ]
    ==
    hybrid_real_test_df[
        "predicted_soil_class"
    ]
)

image_accuracy = image_correct.mean()

print("\nImage accuracy on these test cases:")
print(f"{image_accuracy:.2%}")


# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

hybrid_dir = os.path.join(
    PROJECT_DIR,
    "reports",
    "hybrid"
)

os.makedirs(
    hybrid_dir,
    exist_ok=True
)

final_hybrid_csv = os.path.join(
    hybrid_dir,
    "hybrid_real_validation_results.csv"
)

hybrid_real_test_df.to_csv(
    final_hybrid_csv,
    index=False
)

print("\n✓ Final hybrid validation saved:")
print(final_hybrid_csv)

print("\nRows tested:", len(hybrid_real_test_df))
print("File exists:", os.path.exists(final_hybrid_csv))

In [72]:
# WEEK 4 — SOIL HEALTH SCORE
# Project-level relative soil condition score
# This is NOT an official Soil Health Card score.

import numpy as np
import pandas as pd

print("=" * 70)
print("RELATIVE SOIL HEALTH SCORE")
print("=" * 70)


def calculate_soil_health_score(
    moisture,
    ph,
    organic_matter,
    nitrogen,
    phosphorus,
    potassium
):
    """
    Project-level relative soil condition score.

    The score compares the sample against the observed
    range of the project's cleaned dataset.

    It is NOT an official agronomic or Soil Health Card score.
    """

    # Dataset ranges
    ranges = {
        "moisture": (
            soil_df["moisture"].min(),
            soil_df["moisture"].max()
        ),

        "ph": (
            soil_df["ph"].min(),
            soil_df["ph"].max()
        ),

        "organic_matter": (
            soil_df["organic_matter"].min(),
            soil_df["organic_matter"].max()
        ),

        "nitrogen": (
            soil_df["nitrogen"].min(),
            soil_df["nitrogen"].max()
        ),

        "phosphorus": (
            soil_df["phosphorus"].min(),
            soil_df["phosphorus"].max()
        ),

        "potassium": (
            soil_df["potassium"].min(),
            soil_df["potassium"].max()
        )
    }


    # -----------------------------------------------------
    # Normalize each parameter to 0–100
    # -----------------------------------------------------

    def normalize(value, minimum, maximum):

        if maximum == minimum:
            return 50.0

        score = (
            (value - minimum)
            /
            (maximum - minimum)
        ) * 100

        return float(
            np.clip(score, 0, 100)
        )


    moisture_score = normalize(
        moisture,
        *ranges["moisture"]
    )

    ph_score = normalize(
        ph,
        *ranges["ph"]
    )

    organic_matter_score = normalize(
        organic_matter,
        *ranges["organic_matter"]
    )

    nitrogen_score = normalize(
        nitrogen,
        *ranges["nitrogen"]
    )

    phosphorus_score = normalize(
        phosphorus,
        *ranges["phosphorus"]
    )

    potassium_score = normalize(
        potassium,
        *ranges["potassium"]
    )


    # -----------------------------------------------------
    # Overall score
    # -----------------------------------------------------

    overall_score = np.mean([
        moisture_score,
        ph_score,
        organic_matter_score,
        nitrogen_score,
        phosphorus_score,
        potassium_score
    ])


    # -----------------------------------------------------
    # Project-level interpretation
    # -----------------------------------------------------

    if overall_score >= 75:
        status = "Relatively Healthy"

    elif overall_score >= 50:
        status = "Moderate"

    elif overall_score >= 25:
        status = "Needs Attention"

    else:
        status = "Poor Relative Condition"


    return {
        "soil_health_score": round(
            float(overall_score),
            2
        ),

        "soil_health_status": status,

        "parameter_scores": {
            "moisture": round(moisture_score, 2),
            "ph": round(ph_score, 2),
            "organic_matter": round(
                organic_matter_score, 2
            ),
            "nitrogen": round(
                nitrogen_score, 2
            ),
            "phosphorus": round(
                phosphorus_score, 2
            ),
            "potassium": round(
                potassium_score, 2
            )
        }
    }


print("✓ Soil-health scoring function created")

In [73]:
# FINAL HYBRID ANALYSIS WITH SOIL-HEALTH SCORE

def hybrid_soil_analysis_final(
    image_path,
    moisture,
    ph,
    organic_matter,
    nitrogen,
    phosphorus,
    potassium
):

    # -----------------------------------------------------
    # Image analysis — ResNet-50
    # -----------------------------------------------------

    image = Image.open(image_path).convert("RGB")
    image = image.resize((224, 224))

    image_array = np.asarray(
        image,
        dtype=np.float32
    )

    image_array = np.expand_dims(
        image_array,
        axis=0
    )

    image_array = tf.keras.applications.resnet50.preprocess_input(
        image_array
    )

    features = resnet_base(
        image_array,
        training=False
    )

    probabilities = cnn_classifier.predict(
        features,
        verbose=0
    )[0]

    predicted_index = int(
        np.argmax(probabilities)
    )

    predicted_class = class_names[
        predicted_index
    ]

    cnn_confidence = float(
        probabilities[predicted_index]
    )


    # -----------------------------------------------------
    # XGBoost nutrient analysis
    # -----------------------------------------------------

    X_n = pd.DataFrame(
        [[moisture, ph, organic_matter, phosphorus, potassium]],
        columns=[
            "moisture",
            "ph",
            "organic_matter",
            "phosphorus",
            "potassium"
        ]
    )

    X_p = pd.DataFrame(
        [[moisture, ph, organic_matter, nitrogen, potassium]],
        columns=[
            "moisture",
            "ph",
            "organic_matter",
            "nitrogen",
            "potassium"
        ]
    )

    X_k = pd.DataFrame(
        [[moisture, ph, organic_matter, nitrogen, phosphorus]],
        columns=[
            "moisture",
            "ph",
            "organic_matter",
            "nitrogen",
            "phosphorus"
        ]
    )

    predicted_n = float(
        xgb_n.predict(X_n)[0]
    )

    predicted_p = float(
        xgb_p.predict(X_p)[0]
    )

    predicted_k = float(
        xgb_k.predict(X_k)[0]
    )


    # -----------------------------------------------------
    # Soil-health score
    # -----------------------------------------------------

    health = calculate_soil_health_score(
        moisture=moisture,
        ph=ph,
        organic_matter=organic_matter,
        nitrogen=nitrogen,
        phosphorus=phosphorus,
        potassium=potassium
    )


    # -----------------------------------------------------
    # Final hybrid result
    # -----------------------------------------------------

    return {
        "soil_class": predicted_class,
        "cnn_confidence": cnn_confidence,

        "nitrogen": predicted_n,
        "phosphorus": predicted_p,
        "potassium": predicted_k,

        "soil_health_score":
            health["soil_health_score"],

        "soil_health_status":
            health["soil_health_status"],

        "parameter_scores":
            health["parameter_scores"]
    }


print("=" * 70)
print("FINAL HYBRID FUNCTION")
print("=" * 70)
print("✓ ResNet-50 image analysis")
print("✓ XGBoost N/P/K analysis")
print("✓ Soil-health scoring")
print("✓ Combined hybrid output")
print("=" * 70)

In [74]:
# FINAL HYBRID ANALYSIS WITH SOIL-HEALTH SCORE

def hybrid_soil_analysis_final(
    image_path,
    moisture,
    ph,
    organic_matter,
    nitrogen,
    phosphorus,
    potassium
):

    # -----------------------------------------------------
    # Image analysis — ResNet-50
    # -----------------------------------------------------

    image = Image.open(image_path).convert("RGB")
    image = image.resize((224, 224))

    image_array = np.asarray(
        image,
        dtype=np.float32
    )

    image_array = np.expand_dims(
        image_array,
        axis=0
    )

    image_array = tf.keras.applications.resnet50.preprocess_input(
        image_array
    )

    features = resnet_base(
        image_array,
        training=False
    )

    probabilities = cnn_classifier.predict(
        features,
        verbose=0
    )[0]

    predicted_index = int(
        np.argmax(probabilities)
    )

    predicted_class = class_names[
        predicted_index
    ]

    cnn_confidence = float(
        probabilities[predicted_index]
    )


    # -----------------------------------------------------
    # XGBoost nutrient analysis
    # -----------------------------------------------------

    X_n = pd.DataFrame(
        [[moisture, ph, organic_matter, phosphorus, potassium]],
        columns=[
            "moisture",
            "ph",
            "organic_matter",
            "phosphorus",
            "potassium"
        ]
    )

    X_p = pd.DataFrame(
        [[moisture, ph, organic_matter, nitrogen, potassium]],
        columns=[
            "moisture",
            "ph",
            "organic_matter",
            "nitrogen",
            "potassium"
        ]
    )

    X_k = pd.DataFrame(
        [[moisture, ph, organic_matter, nitrogen, phosphorus]],
        columns=[
            "moisture",
            "ph",
            "organic_matter",
            "nitrogen",
            "phosphorus"
        ]
    )

    predicted_n = float(
        xgb_n.predict(X_n)[0]
    )

    predicted_p = float(
        xgb_p.predict(X_p)[0]
    )

    predicted_k = float(
        xgb_k.predict(X_k)[0]
    )


    # -----------------------------------------------------
    # Soil-health score
    # -----------------------------------------------------

    health = calculate_soil_health_score(
        moisture=moisture,
        ph=ph,
        organic_matter=organic_matter,
        nitrogen=nitrogen,
        phosphorus=phosphorus,
        potassium=potassium
    )


    # -----------------------------------------------------
    # Final hybrid result
    # -----------------------------------------------------

    return {
        "soil_class": predicted_class,
        "cnn_confidence": cnn_confidence,

        "nitrogen": predicted_n,
        "phosphorus": predicted_p,
        "potassium": predicted_k,

        "soil_health_score":
            health["soil_health_score"],

        "soil_health_status":
            health["soil_health_status"],

        "parameter_scores":
            health["parameter_scores"]
    }


print("=" * 70)
print("FINAL HYBRID FUNCTION")
print("=" * 70)
print("✓ ResNet-50 image analysis")
print("✓ XGBoost N/P/K analysis")
print("✓ Soil-health scoring")
print("✓ Combined hybrid output")
print("=" * 70)

In [75]:
# FINAL MILESTONE 2 HYBRID TEST + REPORT

import os
import pandas as pd
from datetime import datetime

final_results = []

# Use 10 real soil-test records and 10 different real test images
num_tests = min(10, len(soil_df), len(test_images))

for i in range(num_tests):

    sample = soil_df.iloc[i]
    image_info = test_images[i]

    result = hybrid_soil_analysis_final(
        image_path=image_info["path"],
        moisture=sample["moisture"],
        ph=sample["ph"],
        organic_matter=sample["organic_matter"],
        nitrogen=sample["nitrogen"],
        phosphorus=sample["phosphorus"],
        potassium=sample["potassium"]
    )

    final_results.append({
        "sample_id": sample["sample_id"],
        "crop": sample["crop"],
        "image": os.path.basename(image_info["path"]),

        "actual_image_class":
            image_info["actual_class"],

        "predicted_soil_class":
            result["soil_class"],

        "cnn_confidence":
            result["cnn_confidence"],

        "input_nitrogen":
            sample["nitrogen"],

        "predicted_nitrogen":
            result["nitrogen"],

        "input_phosphorus":
            sample["phosphorus"],

        "predicted_phosphorus":
            result["phosphorus"],

        "input_potassium":
            sample["potassium"],

        "predicted_potassium":
            result["potassium"],

        "soil_health_score":
            result["soil_health_score"],

        "soil_health_status":
            result["soil_health_status"]
    })


# ---------------------------------------------------------
# Create final table
# ---------------------------------------------------------

final_hybrid_df = pd.DataFrame(final_results)

print("=" * 100)
print("FINAL HYBRID SOIL ANALYSIS")
print("=" * 100)

display(final_hybrid_df)


# ---------------------------------------------------------
# Image classification result
# ---------------------------------------------------------

correct = (
    final_hybrid_df["actual_image_class"]
    ==
    final_hybrid_df["predicted_soil_class"]
)

image_accuracy = correct.mean()

print("\nImage classification accuracy:")
print(f"{image_accuracy:.2%}")

print(
    f"Correct predictions: {correct.sum()}/{len(correct)}"
)


# ---------------------------------------------------------
# Average health score
# ---------------------------------------------------------

average_health_score = (
    final_hybrid_df["soil_health_score"]
    .mean()
)

print("\nAverage relative soil-health score:")
print(f"{average_health_score:.2f}/100")


# ---------------------------------------------------------
# Save final CSV
# ---------------------------------------------------------

hybrid_dir = os.path.join(
    PROJECT_DIR,
    "reports",
    "hybrid"
)

os.makedirs(
    hybrid_dir,
    exist_ok=True
)

final_csv_path = os.path.join(
    hybrid_dir,
    "final_hybrid_soil_assessment.csv"
)

final_hybrid_df.to_csv(
    final_csv_path,
    index=False
)


# ---------------------------------------------------------
# Create final Milestone 2 report
# ---------------------------------------------------------

report_dir = os.path.join(
    PROJECT_DIR,
    "reports",
    "week4"
)

os.makedirs(
    report_dir,
    exist_ok=True
)

report_path = os.path.join(
    report_dir,
    "Milestone_2_Final_Report.md"
)

report = f"""# Milestone 2 — Final Report

## AI-Powered Soil Analytics System

Generated: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

---

## Week 3 — CNN Soil Image Analysis

### Model

ResNet-50 transfer-learning based image analysis was implemented
for seven soil classes:

- Alluvial Soil
- Arid Soil
- Black Soil
- Laterite Soil
- Mountain Soil
- Red Soil
- Yellow Soil

### Dataset

Training images: 794

Validation images: 171

Test images: 174

### Test Performance

Accuracy: 88.51%

Weighted Precision: 88.23%

Weighted Recall: 88.51%

Weighted F1-score: 88.03%

A confusion matrix and incorrect-prediction analysis were completed.

The saved CNN classifier is:

models/cnn/resnet50_soil_classifier.keras

---

## Week 4 — Structured ML

The cleaned soil-test dataset contains 146 records.

The structured parameters are:

- Moisture
- pH
- Organic Matter
- Nitrogen
- Phosphorus
- Potassium

XGBoost regression models were developed for nitrogen,
phosphorus and potassium.

### Final Test Results

#### Nitrogen

MAE: 0.0571

RMSE: 0.1256

R²: 0.4978

#### Phosphorus

MAE: 35.8509

RMSE: 68.4390

R²: 0.4947

#### Potassium

MAE: 227.4419

RMSE: 334.8574

R²: 0.0062

Hyperparameter tuning was performed using GridSearchCV.

The final models were saved under:

models/xgboost/

---

## Explainable AI

### Grad-CAM

Grad-CAM was implemented for the ResNet-50 image-analysis branch.

The visualization identifies image regions contributing to the
CNN prediction, including an incorrect high-confidence prediction.

### SHAP

SHAP explanations were generated for the nitrogen, phosphorus
and potassium XGBoost models.

---

## Hybrid Soil Analysis

The hybrid system combines:

ResNet-50 image analysis

+

XGBoost structured soil-test analysis

The image branch provides:

- Soil class
- CNN confidence

The structured branch provides:

- Nitrogen prediction
- Phosphorus prediction
- Potassium prediction

The combined system also produces a project-level relative
soil-health score.

---

## Hybrid Validation

Number of real records tested: {num_tests}

Image accuracy on this small integration test: {image_accuracy:.2%}

Average relative soil-health score: {average_health_score:.2f}/100

The detailed results are available in:

reports/hybrid/final_hybrid_soil_assessment.csv

Important limitation:

The available image dataset and structured soil-test dataset do
not provide a verified one-to-one correspondence between individual
images and individual soil-test samples. Therefore, this validation
demonstrates pipeline integration rather than scientifically paired
multimodal accuracy.

---

## Soil-Health Score

The implemented score is a project-level relative score based on
the six parameters available in the cleaned dataset.

It compares parameter values with the observed ranges in the
project dataset.

It is NOT an official Soil Health Card score and should not be
interpreted as a validated agronomic percentage of soil health.

---

## Final Models and Methods

- ResNet-50 — soil image analysis
- Dense classifier head — soil classification
- XGBoost — N/P/K regression
- Grad-CAM — CNN explainability
- SHAP — structured-model explainability
- Hybrid ResNet-50 + XGBoost — combined soil assessment
- Relative soil-health scoring — project-level assessment

---

## Milestone 2 Deliverables

✓ CNN model

✓ CNN evaluation

✓ Confusion matrix

✓ Incorrect prediction analysis

✓ Grad-CAM

✓ XGBoost nitrogen model

✓ XGBoost phosphorus model

✓ XGBoost potassium model

✓ XGBoost evaluation

✓ Hyperparameter tuning

✓ SHAP explanations

✓ Hybrid analysis

✓ Hybrid validation

✓ Soil-health scoring implementation

✓ Final Milestone 2 report

---

## Conclusion

The machine-learning, explainability and hybrid-analysis components
specified for Milestone 2 have been implemented using the available
project data.

The system combines visual soil classification with structured
soil-test analysis and provides a project-level relative soil-health
assessment.

The absence of paired image/soil-test records and validated
soil-health thresholds is documented as a project limitation.
"""

with open(
    report_path,
    "w",
    encoding="utf-8"
) as f:
    f.write(report)


print("\n" + "=" * 100)
print("MILESTONE 2 FINAL FILES")
print("=" * 100)

print("✓ Hybrid results:")
print(final_csv_path)

print("✓ Final report:")
print(report_path)

print("\nFiles exist:")
print("Hybrid CSV:", os.path.exists(final_csv_path))
print("Final report:", os.path.exists(report_path))

print("\n" + "=" * 100)
print("MILESTONE 2 IMPLEMENTATION COMPLETE")
print("=" * 100)